<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap09/cap09.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Pratica con Esercizi di Programmazione**

La presente lista di **Esercizi di Programmazione (EP)** consolida le formulazioni teoriche presentate nel Capitolo 9 — Deep Learning per la Visione Artificiale — tramite un percorso pratico applicato. Diversamente dall'addestramento di reti neurali complete con PyTorch, che richiede tempi di esecuzione e, talvolta, GPU, gli EP di questo capitolo isolano le **grandezze intermedie** di una *pipeline* reale di deep learning — l'output di un singolo strato convoluzionale, il risultato di un'operazione di *pooling*, il conteggio dei parametri addestrabili di un'architettura, la sovrapposizione tra bounding box candidate, la qualità di una maschera di segmentazione e il filtro di soppressione non-massima — consentendo di validare manualmente ogni fase del ragionamento senza dipendere da librerie di machine learning né da un addestramento reale.

L'incatenamento degli esercizi riproduce il flusso concettuale del capitolo e cresce in difficoltà a ogni passo: si inizia con il calcolo manuale dell'output di uno **strato convoluzionale addestrato** (🟢), a partire da un *kernel* e un bias già addestrati; si prosegue con l'operazione di ***pooling*** (🟢, massimo e media), che riduce la risoluzione spaziale tra blocchi convoluzionali; si continua con il **conteggio dei parametri addestrabili** (🟡) di un'architettura CNN completa, evidenziando perché la condivisione dei pesi rende queste reti così più economiche rispetto a uno strato completamente connesso equivalente; si approfondisce il calcolo dell'**Intersezione su Unione (IoU)** e della **Soppressione Non-Massima (NMS)** (🟡), fase di post-elaborazione comune a rilevatori come Faster R-CNN e YOLO; si passa alla **valutazione delle maschere di segmentazione** (🟠) con le stesse metriche IoU e Dice utilizzate per confrontare U-Net con la baseline morfologica classica; e si conclude con una ***pipeline* integrata** (🔴), unendo l'output di un rilevatore di oggetti (dopo NMS) a una misurazione del mondo reale tramite riferimento di scala — lo stesso principio della fotogrammetria studiato nell'integrazione finale del capitolo.

Ogni volta che ha senso, ciascun esercizio indica i metodi della libreria didattica `morph.py` (la stessa utilizzata nel capitolo, importata come `mm`) che risolvono una fase del problema o che servono da riferimento per verificare i propri calcoli — senza, tuttavia, sostituire il ragionamento che devi implementare.

> ### ❗ Linee Guida per la Risoluzione degli Esercizi di Programmazione
>
> In tutti gli esercizi di questo capitolo, le fasi di discretizzazione o arrotondamento numerico devono impiegare l'arrotondamento standard all'intero più vicino (*round half away from zero*), mitigando le ambiguità in valori con frazione esattamente uguale a $0{,}5$. Salvo indicazione esplicita contraria: (i) l'operazione di "convoluzione" segue la convenzione adottata dai *framework* di deep learning — **correlazione incrociata**, senza inversione spaziale del *kernel*, esattamente come presentato nella Sezione "Strato Convoluzionale"; (ii) il riempimento (*padding*) è effettuato con zeri; (iii) le bounding box sono specificate nel formato angolo-a-angolo $(x_1, y_1, x_2, y_2)$, con $x_1 < x_2$ e $y_1 < y_2$; e (iv) vettori/matrici seguono l'indicizzazione a partire da $0$, con la convenzione `[riga][colonna]` per strutture bidimensionali.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EPs)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

#### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella sottostante:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Esecuzione dei Test
Per valutare i test, esegui `TestSuite("EP09_01.extensão").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola automaticamente il voto.

Per testare direttamente il codice Python, senza salvare il file, usa `run_code(codigo)` passando il codice come *stringa* in una variabile `codigo`:

```python
codigo = """
# ... il tuo codice qui ...
"""
TestSuite("EP09_01").run_code(codigo)
```

### EP09_01 🟢 Convoluzione 2D Manuale (*Forward* di un Livello Appreso)

Il PyTorch, presentato in questo capitolo, esegue `nn.Conv2d(x)` in un'unica chiamata — ma dietro di essa c'è solo la correlazione incrociata tra un *kernel* (già addestrato) e un intorno dell'input, seguita dalla somma di un bias e di un'attivazione, esattamente come formalizzato nella Sezione "Livello Convoluzionale". La differenza essenziale rispetto alla convoluzione con *kernel* fissi del Capitolo 3 è che, qui, i valori del *kernel* e del bias **sono già pronti** (come se fossero stati appresi per gradiente), e spetta a te riprodurre manualmente il passaggio diretto (*forward pass*) che il *framework* esegue internamente.

Prima di addestrare una vera CNN, ti è stato affidato il compito di implementare questo passaggio diretto da zero, per un singolo livello convoluzionale con un singolo canale di input e un singolo filtro di output, incluso il supporto a *padding* e *stride* arbitrari.

#### 📋 Linee Guida di Implementazione

1. **Input:** Leggere le dimensioni $H \times W$ della mappa delle caratteristiche di input e, successivamente, i suoi $H \times W$ valori reali.
   
2. ***Kernel* e bias:** Leggere le dimensioni $k_h \times k_w$ del *kernel* (già addestrato), i suoi valori reali, e il bias $b$ (reale, scalare).
   
3. **Iperparametri:** Leggere il *padding* $p$ (intero, numero di zeri aggiunti su ciascun bordo) e lo *stride* $s$ (intero, passo dello scorrimento).
   
4. **Riempimento:** Aggiungere $p$ zeri su ciascuno dei quattro bordi della mappa di input prima della correlazione.
   
5. **Correlazione incrociata:** Per ogni posizione di output $(i, j)$, calcolare
   $$
   z(i,j) = b + \sum_{u=0}^{k_h-1} \sum_{v=0}^{k_w-1} K(u,v) \cdot X_{pad}(i \cdot s + u,\; j \cdot s + v),
   $$
   scorrendo l'input **senza** invertire il *kernel* (convenzione dei *framework* di deep learning, diversa dalla convoluzione matematica classica).

6. **Attivazione:** Applicare ReLU a ogni valore: $a(i,j) = \max(0, z(i,j))$.

7. **Dimensioni di output:** $O_h = \lfloor (H + 2p - k_h)/s \rfloor + 1$ e $O_w = \lfloor (W + 2p - k_w)/s \rfloor + 1$.

8. **Output:** Stampare $O_h$ e $O_w$ nella prima riga, seguiti da $O_h$ righe con $O_w$ valori reali ciascuna (la mappa delle caratteristiche di output, già con ReLU applicata), formattati con 4 cifre decimali.

#### 📌 Vincoli Computazionali

* **Un canale di input, un filtro di output:** non è necessario gestire più canali o più filtri in questa versione semplificata.
* **Senza inversione del *kernel*:** implementare la correlazione incrociata, non la convoluzione matematica classica con *kernel* invertito — è questa l'operazione che PyTorch (e la maggior parte dei *framework*) chiama "convoluzione".
* **Riempimento con zeri:** i $p$ pixel aggiunti su ciascun bordo valgono sempre $0$.
* **Formattazione:** tutti i valori di output devono avere esattamente 4 cifre decimali, anche quando il valore è un intero (es.: `2.0000`).

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo nel livello convoluzionale |
|---|---|
| *Kernel* $K$ | Parametri appresi per gradiente, analoghi ai coefficienti di un filtro fisso del Capitolo 3, ma regolati tramite backpropagation |
| Bias $b$ | Offset appreso, sommato dopo la correlazione — consente al neurone di "attivarsi" anche con input nullo |
| *Padding* | Controlla la dimensione spaziale dell'output e previene la perdita di informazioni ai bordi a ogni livello |
| *Stride* | Controlla il passo dello scorrimento; valori $> 1$ riducono la risoluzione spaziale, come una forma di sottocampionamento integrato nella convoluzione stessa |
| ReLU | Introduce non linearità dopo la combinazione lineare, esattamente come nella Sezione "Funzione di Attivazione" |

#### 🧩 Metodi di `morph.py` che possono aiutare

* `mm.readImg(h, w, dtype='float')` — legge direttamente una matrice $h \times w$ di valori reali dall'input standard, evitando il *parsing* manuale della mappa delle caratteristiche e del *kernel*.
* `mm.correlacao0(f, kernel, bias)` — implementa la stessa somma di correlazione incrociata + bias che dovrai calcolare a mano, ma **senza** supporto per *padding* o *stride*, e converte il risultato in `uint8` (tronca valori negativi e decimali). Può servire come riferimento concettuale o per verificare il caso più semplice ($p=0$, $s=1$), ma non sostituisce la tua implementazione completa — che deve preservare segno, cifre decimali, *padding*, *stride* e ReLU.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ valori reali ciascuna (mappa di input).
* Riga successiva: Interi $k_h$ e $k_w$.
* Prossime $k_h$ righe: $k_w$ valori reali ciascuna (*kernel*).
* Riga successiva: Reale $b$ (bias).
* Riga successiva: Interi $p$ e $s$.

**Output:**

* Riga 1: Interi $O_h$ e $O_w$.
* Prossime $O_h$ righe: $O_w$ valori reali ciascuna, con 4 cifre decimali.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 1<br>1 1<br>-2<br>0 1 | 2 2<br>2.0000 3.0000<br>0.0000 2.0000 | *Padding* 0, *stride* 1: output $2\times2$ senza riempimento. |
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 0<br>0 1<br>0<br>1 2 | 2 2<br>1.0000 0.0000<br>1.0000 2.0000 | *Padding* 1, *stride* 2: input riempito con zeri prima della correlazione. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0901" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Convoluzione 2D Manuale</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 correlazione incrociata + bias + ReLU</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ingresso 4×4 fisso, kernel 2×2 fisso (evidenziato in blu) &mdash; regola <em>padding</em> (p), <em>stride</em> (s) e bias (b), esattamente i parametri che l'EP09_01 richiede in ingresso, e osserva come cambiano la dimensione e i valori dell'uscita.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Padding (p)</div>
        <div id="ep0901_pad_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0901_stride_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Bias (b)</div>
        <input id="ep0901_bias" type="number" step="0.5" value="0.5" style="width:70px;font-family:monospace;text-align:center;border:1px solid #ccc;border-radius:6px;padding:3px;">
      </div>
    </div>

    <div id="ep0901_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Posizione di uscita (i,j)</label>
        <span id="ep0901_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0901_sl" style="width:100%;accent-color:#2980b9;" max="8" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Ingresso X imbottito (con padding)</div>
        <div id="ep0901_grid" style="display:grid;gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> originale</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#f5f5f5;border:1px dashed #ccc;border-radius:2px;vertical-align:middle;"></span> padding (0)</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> finestra corrente</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Kernel K (2×2)</div>
        <div id="ep0901_kernel" style="display:grid;grid-template-columns:repeat(2,44px);gap:3px;"></div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Uscita Y = ReLU(X⊛K + b)</div>
        <div id="ep0901_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0901_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ riavvia l'esplorazione</button>
    </div>
    <div id="ep0901_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Ogni posizione dello slider rivela una cella della matrice di uscita. Percorri tutte le posizioni per completare la mappa di uscita. Cambiare p, s o b riavvia l'esplorazione, perché la mappa di uscita cambia dimensione e/o valori.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4, kh = 2, kw = 2;
    var X = [[1,3,2,0],[0,1,4,1],[2,0,1,3],[1,2,0,1]];
    var K = [[1,0],[0,-1]];

    var state = { p: 0, s: 1, bias: 0.5 };
    var visited = {};

    var slEl = root.querySelector('#ep0901_sl');
    var vlEl = root.querySelector('#ep0901_vl');
    var gridEl = root.querySelector('#ep0901_grid');
    var kernelEl = root.querySelector('#ep0901_kernel');
    var outEl = root.querySelector('#ep0901_out');
    var dbg = root.querySelector('#ep0901_debug');
    var formulaEl = root.querySelector('#ep0901_formula');
    var resetBtn = root.querySelector('#ep0901_reset');
    var padBtnsEl = root.querySelector('#ep0901_pad_btns');
    var strideBtnsEl = root.querySelector('#ep0901_stride_btns');
    var biasInput = root.querySelector('#ep0901_bias');

    kernelEl.innerHTML = '';
    for(var u=0; u<kh; u++) for(var v=0; v<kw; v++){
      var kd = document.createElement('div');
      kd.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;background:#bbdefb;border:1px solid #64b5f6;border-radius:6px;font-family:monospace;font-weight:bold;color:#0d47a1;';
      kd.textContent = K[u][v];
      kernelEl.appendChild(kd);
    }

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function buildPadded(p){
      var size = H + 2*p;
      var Xp = [];
      for(var r=0; r<size; r++){
        var row = [];
        for(var c=0; c<size; c++){
          var origR = r-p, origC = c-p;
          var isPad = !(origR>=0 && origR<H && origC>=0 && origC<W);
          row.push({ val: isPad ? 0 : X[origR][origC], pad: isPad });
        }
        Xp.push(row);
      }
      return Xp;
    }

    function computeAll(p, s, bias){
      var Xp = buildPadded(p);
      var size = H + 2*p;
      var Oh = Math.floor((size - kh)/s) + 1;
      var Ow = Math.floor((size - kw)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var soma = 0;
          for(var u=0; u<kh; u++) for(var v=0; v<kw; v++) soma += K[u][v]*Xp[i*s+u][j*s+v].val;
          var z = soma + bias;
          var a = Math.max(0, z);
          vals[i].push({ soma: soma, z: z, a: a });
        }
      }
      return { Xp: Xp, size: size, Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.p, state.s, state.bias);
      gridEl.style.gridTemplateColumns = 'repeat(' + model.size + ', ' + Math.min(44, Math.floor(360/model.size)) + 'px)';
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 44px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' + 2·' + state.p + ' − ' + kh + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' + 2·' + state.p + ' − ' + kw + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;

      gridEl.innerHTML = '';
      var cellPx = Math.min(44, Math.floor(360/model.size));
      for(var r=0; r<model.size; r++){
        for(var c=0; c<model.size; c++){
          var cell = model.Xp[r][c];
          var dentroJanela = (r>=winRowStart && r<winRowStart+kh && c>=winColStart && c<winColStart+kw);
          var d = document.createElement('div');
          var base = 'width:'+cellPx+'px;height:'+cellPx+'px;display:flex;align-items:center;justify-content:center;border-radius:5px;font-family:monospace;font-size:11px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(cell.pad){
            base += 'background:#f5f5f5;border:1px dashed #ccc;color:#bbb;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = cell.val;
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          var isCurrent = (oi===i && oj===j);
          var wasVisited = !!visited[oi+','+oj];
          var od = document.createElement('div');
          var style = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi][oj].a.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela em ('+i+','+j+'), topo-esquerda em X_pad('+winRowStart+','+winColStart+')  |  soma(X⊙K)='+cur.soma.toFixed(2)+'  +  viés='+state.bias.toFixed(2)+'  =  z='+cur.z.toFixed(2)+'  →  ReLU(z)='+cur.a.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setPadding(val){
      state.p = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    function setStride(val){
      state.s = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
    buildButtons(strideBtnsEl, [1,2], state.s, setStride);

    biasInput.addEventListener('change', function(){
      var v = parseFloat(biasInput.value);
      state.bias = isNaN(v) ? 0 : v;
      rebuildModel(true);
    });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0901');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.1:** Simulatore EP09_01: Convoluzione 2D Manuale (correlazione incrociata + bias + ReLU, con *padding* e *stride* regolabili)


<figure id="fig-09-sim-ep0901">
  <img src="imagens/fig-09-sim-ep0901.png" alt=" Simulatore EP09_01: Convoluzione 2D Manuale (correlazione incrociata + bias + ReLU, con *padding* e *stride* regolabili) " style="max-width:80%" />
  <figcaption><strong>Figura 9.1:</strong>  Simulatore EP09_01: Convoluzione 2D Manuale (correlazione incrociata + bias + ReLU, con *padding* e *stride* regolabili) </figcaption>
</figure>

In [ ]:
%%writefile EP09_01.py
# Codice Python

In [ ]:
TestSuite("EP09_01.py").run()

### EP09_02 🟢 *Pooling* Manuale (Massimo e Media)

Tra i blocchi convoluzionali, l'architettura tipica di una CNN intercala livelli di ***pooling***, che riducono la risoluzione spaziale della mappa delle caratteristiche senza introdurre nuovi parametri addestrabili — a differenza della convoluzione, il *pooling* non ha pesi: si limita a riassumere ogni finestra dell'ingresso in un singolo valore, tramite un massimo o una media, esattamente come formalizzato nella Sezione "*Pooling*".

Sei stato incaricato di implementare questa operazione a partire da una finestra scorrevole quadrata, senza sovrapposizione parziale sui bordi (solo finestre complete), supportando i due tipi più comuni: `max` (preserva il valore più saliente, tipicamente usato per mantenere bordi e texture forti) e `avg` (smussa la regione, preservando l'informazione di intensità media).

#### 📋 Linee Guida di Implementazione

1. **Ingresso:** Leggere le dimensioni $H \times W$ della mappa delle caratteristiche di ingresso e i suoi $H \times W$ valori reali.
2. **Finestra:** Leggere gli interi $k$ (dimensione della finestra quadrata $k \times k$) e $s$ (*stride*).
3. **Tipo:** Leggere una *stringa*, `max` o `avg`, che indica il tipo di *pooling*.
4. **Senza riempimento:** Questa operazione **non** utilizza *padding*; le finestre che supererebbero il bordo dell'ingresso vengono scartate.
5. **Calcolo:** Per ogni posizione di uscita $(i,j)$, calcolare il massimo o la media dei $k \times k$ valori della finestra corrispondente, iniziando da $(i \cdot s,\, j \cdot s)$.
6. **Dimensioni di uscita:** $O_h = \lfloor (H - k)/s \rfloor + 1$ e $O_w = \lfloor (W - k)/s \rfloor + 1$.
7. **Uscita:** Stampare $O_h$ e $O_w$ nella prima riga, seguiti da $O_h$ righe con $O_w$ valori reali ciascuna, formattati con 4 cifre decimali.

#### 📌 Vincoli Computazionali

* **Finestra quadrata:** $k \times k$, senza supporto per finestre rettangolari in questa versione.
* **Senza *padding*:** solo le finestre interamente contenute nell'ingresso sono considerate — le dimensioni che "avanzeranno" sono semplicemente scartate.
* **`avg` usa divisione reale:** la media è sempre $\text{somma}/k^2$, anche quando il risultato ha molte cifre decimali — arrotondare solo nella formattazione finale, secondo la linea guida generale del capitolo.
* **Formattazione:** tutti i valori di uscita con esattamente 4 cifre decimali.

#### 🧠 Fondamento Teorico

| Elemento | Ruolo nell'architettura |
|---|---|
| *Pooling* massimo | Preserva l'attivazione più forte della finestra; comune dopo livelli convoluzionali per mantenere bordi e texture salienti |
| *Pooling* medio | Smussa la regione, preservando l'intensità media; comune nei livelli finali (*global average pooling*) |
| Assenza di parametri | Differenzia il *pooling* dalla convoluzione: riduce la risoluzione spaziale senza costi aggiuntivi di addestramento |
| Riduzione della risoluzione | Contribuisce all'invarianza rispetto a piccole traslazioni e alla riduzione del costo computazionale dei livelli successivi |

#### 🧩 Metodi di `morph.py` che possono aiutare

Il `morph.py` non implementa il *pooling* con sottocampionamento direttamente, ma due famiglie di operazioni mostrano la stessa idea sotto un'altra ottica, utile per verificare la tua intuizione:

* `mm.dil(f, Bc)` / `mm.dil0(f, B)` — dilatazione morfologica: sostituisce ogni pixel con il **massimo** del suo intorno definito dall'elemento strutturante $B$ (es.: `mm.sebox(n)` per una finestra $(2n+1)\times(2n+1)$). È concettualmente un "*max-pooling* senza sottocampionamento" (produce un'immagine della stessa dimensione, invece che ridotta).
* `mm.blur(f, N)` — smussatura per media in una finestra $N \times N$, analoga all'*avg-pooling*, anch'essa senza riduzione della risoluzione.
* `mm.readImg(h, w, dtype='float')` — utile per leggere la mappa di ingresso in virgola mobile.

#### 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ valori reali ciascuna.
* Riga successiva: Interi $k$ e $s$.
* Riga successiva: `max` o `avg`.

**Uscita:**

* Riga 1: Interi $O_h$ e $O_w$.
* Prossime $O_h$ righe: $O_w$ valori reali ciascuna, con 4 cifre decimali.

#### 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>max | 2 2<br>6.0000 4.0000<br>4.0000 5.0000 | *Pooling* massimo, finestra $2\times2$, *stride* 2. |
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>avg | 2 2<br>3.7500 2.2500<br>2.2500 2.2500 | *Pooling* medio sulle stesse finestre. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0902" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Pooling Manuale</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 senza padding, finestre complete</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ingresso 4×4 fisso &mdash; regola la dimensione della finestra (k), lo stride (s) e il tipo, esattamente i parametri che EP09_02 legge in ingresso, e guarda come cambiano la dimensione e i valori dell'uscita.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Finestra (k)</div>
        <div id="ep0902_k_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0902_s_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Tipo</div>
        <div style="display:flex;gap:8px;">
          <button id="ep0902_max" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #f0ad4e;background:#fff3cd;color:#7a5c00;font-weight:bold;font-size:11px;">max</button>
          <button id="ep0902_avg" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #ddd;background:#f3f4f6;color:#555;font-weight:bold;font-size:11px;">avg</button>
        </div>
      </div>
    </div>

    <div id="ep0902_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Posizione di uscita (i,j)</label>
        <span id="ep0902_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0902_sl" style="width:100%;accent-color:#2980b9;" max="3" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Ingresso X (4×4)</div>
        <div id="ep0902_grid" style="display:grid;grid-template-columns:repeat(4,44px);gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> fuori dalla finestra</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> finestra corrente</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#eee;border:1px dashed #bbb;border-radius:2px;vertical-align:middle;"></span> scartato (avanzo)</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Uscita Y (pooling)</div>
        <div id="ep0902_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0902_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ riavvia esplorazione</button>
    </div>
    <div id="ep0902_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Ogni posizione dello slider rivela una cella della matrice di uscita. Le celle grigio tratteggiate nell'ingresso sono "avanzi" che nessuna finestra raggiunge &mdash; nota come ciò accade quando (H&minus;k) non è multiplo di s. Cambiare k, s o il tipo riavvia l'esplorazione.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4;
    var X = [[1,3,2,4],[5,6,1,2],[2,1,0,3],[4,2,5,1]];

    var state = { k: 2, s: 2, tipo: 'max' };
    var visited = {};

    var slEl = root.querySelector('#ep0902_sl');
    var vlEl = root.querySelector('#ep0902_vl');
    var gridEl = root.querySelector('#ep0902_grid');
    var outEl = root.querySelector('#ep0902_out');
    var dbg = root.querySelector('#ep0902_debug');
    var formulaEl = root.querySelector('#ep0902_formula');
    var resetBtn = root.querySelector('#ep0902_reset');
    var kBtnsEl = root.querySelector('#ep0902_k_btns');
    var sBtnsEl = root.querySelector('#ep0902_s_btns');
    var btnMax = root.querySelector('#ep0902_max');
    var btnAvg = root.querySelector('#ep0902_avg');

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function estiloTipoBotoes(){
      btnMax.style.background = state.tipo==='max' ? '#fff3cd' : '#f3f4f6';
      btnMax.style.borderColor = state.tipo==='max' ? '#f0ad4e' : '#ddd';
      btnMax.style.color = state.tipo==='max' ? '#7a5c00' : '#555';
      btnAvg.style.background = state.tipo==='avg' ? '#fff3cd' : '#f3f4f6';
      btnAvg.style.borderColor = state.tipo==='avg' ? '#f0ad4e' : '#ddd';
      btnAvg.style.color = state.tipo==='avg' ? '#7a5c00' : '#555';
    }

    function computeAll(k, s, tipo){
      var Oh = Math.floor((H - k)/s) + 1;
      var Ow = Math.floor((W - k)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var janela = [];
          for(var r=i*s; r<i*s+k; r++) for(var c=j*s; c<j*s+k; c++) janela.push(X[r][c]);
          var resultado = tipo === 'max'
            ? Math.max.apply(null, janela)
            : janela.reduce(function(a,b){return a+b;},0)/janela.length;
          vals[i].push({ janela: janela, resultado: resultado });
        }
      }
      return { Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.k, state.s, state.tipo);
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 48px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      estiloTipoBotoes();
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;
      var alcancavel = []; // marca quais células de X são alcançadas por ALGUMA janela válida
      for(var r=0;r<H;r++){ alcancavel.push(new Array(W).fill(false)); }
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          for(var r=oi*state.s; r<oi*state.s+state.k; r++)
            for(var c=oj*state.s; c<oj*state.s+state.k; c++)
              alcancavel[r][c] = true;
        }
      }

      gridEl.innerHTML = '';
      for(var r=0; r<H; r++){
        for(var c=0; c<W; c++){
          var dentroJanela = (r>=winRowStart && r<winRowStart+state.k && c>=winColStart && c<winColStart+state.k);
          var d = document.createElement('div');
          var base = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(!alcancavel[r][c]){
            base += 'background:#eee;border:1px dashed #bbb;color:#999;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = X[r][c];
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi2=0; oi2<model.Oh; oi2++){
        for(var oj2=0; oj2<model.Ow; oj2++){
          var isCurrent = (oi2===i && oj2===j);
          var wasVisited = !!visited[oi2+','+oj2];
          var od = document.createElement('div');
          var style = 'width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi2][oj2].resultado.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela=['+cur.janela.join(', ')+']  |  tipo='+state.tipo+'  →  resultado='+cur.resultado.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setK(val){
      state.k = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    function setS(val){
      state.s = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
    buildButtons(sBtnsEl, [1,2,3], state.s, setS);

    btnMax.addEventListener('click', function(){ state.tipo='max'; rebuildModel(true); });
    btnAvg.addEventListener('click', function(){ state.tipo='avg'; rebuildModel(true); });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0902');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.2:** Simulatore EP09_02: Pooling Manuale (massimo vs. media, con finestra k e stride s regolabili)


<figure id="fig-09-sim-ep0902">
  <img src="imagens/fig-09-sim-ep0902.png" alt=" Simulatore EP09_02: Pooling Manuale (massimo vs. media, con finestra k e stride s regolabili) " style="max-width:80%" />
  <figcaption><strong>Figura 9.2:</strong>  Simulatore EP09_02: Pooling Manuale (massimo vs. media, con finestra k e stride s regolabili) </figcaption>
</figure>

In [ ]:
%%writefile EP09_02.py
# Codice Python

In [ ]:
TestSuite("EP09_02.py").run()

### EP09_03 🟡 Conteggio dei Parametri Addestrabili di una CNN

Questo EP formalizza il conteggio dei parametri addestrabili di una *CNN*. Data la descrizione testuale di una piccola architettura, composta da layer convoluzionali, di *pooling* e completamente connessi, determinare, per ogni layer, il numero di parametri addestrabili e il totale della rete.

L'architettura deve essere interpretata **sequenzialmente**: l'uscita di un layer convoluzionale diventa l'ingresso del layer successivo compatibile. Pertanto, il numero di canali prodotti da un layer `CONV` determina il numero di canali di ingresso (`cin`) del layer convoluzionale successivo.

In un layer convoluzionale, è importante distinguere **canali di ingresso** e **canali di uscita**:

* $c_{in}$ (*channels in*) è il numero di **canali che entrano nel layer**. Un'immagine in scala di grigi ha $c_{in}=1$, mentre un'immagine RGB ha $c_{in}=3$. In un layer convoluzionale intermedio, `cin` è normalmente uguale al numero di canali prodotti dal layer `CONV` precedente.
* $c_{out}$ (*channels out*) è il numero di **canali prodotti dal layer**. È uguale al numero di filtri utilizzati. Pertanto, se un layer ha 16 filtri, produce $c_{out}=16$ canali.

Ad esempio, si consideri la sequenza:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
POOL
FC 784 10 1
```

La prima convoluzione riceve un'immagine con un canale e produce 8 canali. Dopo il *pooling*, la seconda convoluzione riceve questi 8 canali e ne produce 16. Il layer `POOL` non altera il numero di canali, può solo ridurre le dimensioni spaziali. Il layer `FC` riceve la quantità di ingressi indicata nella propria descrizione.

Ogni filtro convoluzionale ha dimensioni

$$
k_h \times k_w \times c_{in}.
$$

Pertanto, un layer con $c_{out}$ filtri ha

$$
k_h \cdot k_w \cdot c_{in} \cdot c_{out}
$$

pesi. Se c'è un bias, si aggiunge un parametro per ogni filtro, totalizzando altri $c_{out}$ parametri.

Il punto centrale di questo esercizio è osservare che la quantità di parametri di un layer convoluzionale **non dipende dalle dimensioni spaziali** ($H \times W$) della mappa delle caratteristiche. Ciò è dovuto alla **condivisione dei pesi**: lo stesso filtro viene riutilizzato in diverse posizioni dell'ingresso.

#### 📋 Linee Guida di Implementazione

1. **Ingresso:** Leggere l'intero $L$ (numero di layer dell'architettura, nell'ordine in cui vengono applicati).

2. **Layer:** Leggere $L$ righe, ciascuna descrive un layer in uno dei tre formati:

   * `CONV kh kw cin cout bias` — layer convoluzionale con *kernel* $k_h \times k_w$, $c_{in}$ canali di ingresso, $c_{out}$ canali di uscita e `bias` (0 o 1), che indica se c'è un bias per filtro;
   * `POOL` — layer di *pooling* (massimo o medio), che non ha parametri addestrabili e preserva il numero di canali;
   * `FC in out bias` — layer completamente connesso con `in` ingressi, `out` uscite e `bias` (0 o 1), che indica se c'è un bias per neurone.

3. **Coerenza tra layer `CONV`:** in una sequenza di layer convoluzionali, il `cin` di un layer deve corrispondere al `cout` del layer convoluzionale precedente. Un layer `POOL` non altera questo numero di canali.

   Ad esempio:

   ```text
   CONV 3 3 1 8 1
   POOL
   CONV 3 3 8 16 1
   ```

   La prima `CONV` produce 8 canali, che vengono ricevuti dalla seconda `CONV`. Pertanto, nel secondo layer, `cin=8` e `cout=16`.

4. **Parametri di un layer `CONV`:**

   Ciascuno dei $c_{out}$ filtri ha $k_h \cdot k_w \cdot c_{in}$ pesi. Pertanto,

   $$
   P_{\mathrm{CONV}} =
   k_h \cdot k_w \cdot c_{in} \cdot c_{out}
   +
   c_{out}\cdot\text{bias}.
   $$

5. **Parametri di un layer `FC`:**

   $$
   P_{\mathrm{FC}} = 
   \text{in}\cdot\text{out}
   +
   \text{out}\cdot\text{bias}.
   $$

6. **Parametri di un layer `POOL`:** sempre $0$.

7. **Totale della rete:** sommare i parametri addestrabili di tutti i layer.

8. **Uscita:** Per ogni layer, nell'ordine di lettura, stampare `Camada i: P`, dove $i$ inizia da $1$ e $P$ è il numero di parametri di quel layer. Alla fine, stampare `Total: T`.

#### 📐 Esempio per capire `cin` e `cout`

Si consideri la sequenza:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
```

Nel primo layer:

* `cin=1`: entra un canale;
* `cout=8`: ci sono 8 filtri e, quindi, escono 8 canali.

Ogni filtro ha

$$
3\cdot3\cdot1=9
$$

pesi. Poiché ci sono 8 filtri:

$$
9\cdot8=72
$$

pesi. Con un bias per filtro:

$$
72+8=80.
$$

Nel secondo layer:

* `cin=8`: entrano gli 8 canali prodotti dalla prima `CONV`;
* `cout=16`: ci sono 16 filtri e, quindi, escono 16 canali.

Ogni filtro ha

$$
3\cdot3\cdot8=72
$$

pesi. Poiché ci sono 16 filtri:

$$
72\cdot16=1152
$$

pesi. Con 16 bias:

$$
1152+16=1168.
$$

Pertanto, i due layer hanno, rispettivamente, **80** e **1168 parametri addestrabili**.

Si noti che `cout` **non è** $cin$ moltiplicato per il numero di filtri. Il numero di filtri è esattamente `cout`: ogni filtro combina tutti i canali di ingresso e produce **un singolo canale di uscita**.

#### 📌 Vincoli Computazionali

* **Indipendenza dalla dimensione spaziale:** l'ingresso non fornisce $H \times W$. Il conteggio di un layer `CONV` dipende solo da `kh`, `kw`, `cin` e `cout`.
* **Coerenza dei canali:** per due layer `CONV` consecutivi, il `cin` del secondo deve essere uguale al `cout` del primo. Un layer `POOL` preserva il numero di canali.
* **`bias` sempre 0 o 1:** moltiplicare direttamente il termine di bias per questo valore.
* **Layer `POOL` senza argomenti aggiuntivi:** la riga contiene solo la parola `POOL`.
* **Layer `FC`:** il numero di ingressi `in` è fornito esplicitamente. Non è necessario calcolare le dimensioni spaziali prodotte dai layer precedenti.
* Tutti i valori numerici di ingresso sono interi non negativi.

#### 🧠 Fondamenti Teorici

| Elemento                  | Ruolo nel conteggio dei parametri                                                                               |
| ------------------------- | -------------------------------------------------------------------------------------------------------------- |
| $c_{in}$                  | Numero di canali ricevuti dal layer                                                                             |
| $c_{out}$                 | Numero di filtri e, quindi, di canali prodotti dal layer                                                        |
| Filtro convoluzionale     | Ogni filtro ha $k_h \cdot k_w \cdot c_{in}$ pesi e produce un canale di uscita                                 |
| Condivisione dei pesi     | Lo stesso filtro viene riutilizzato in diverse posizioni dell'ingresso, rendendo il conteggio indipendente da $H \times W$ |
| Bias                      | Un singolo parametro aggiuntivo per filtro (`CONV`) o per neurone (`FC`)                                        |
| *Pooling*                 | Può alterare $H \times W$, ma non ha parametri addestrabili e preserva il numero di canali                      |
| Layer `FC`                | Ha un peso per ogni combinazione tra ingresso e neurone di uscita                                               |

#### 🧩 Metodi di `morph.py` che possono aiutare

Questo esercizio è puramente aritmetico e non utilizza direttamente le funzioni di `morph.py`. Il conteggio può, tuttavia, essere verificato in un'architettura reale implementata in PyTorch tramite:

```python
sum(p.numel() for p in modelo.parameters())
```

Questa espressione calcola i parametri del modello, inclusi pesi e bias.

#### 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $L$.
* Prossime $L$ righe: descrizione di ogni layer, nel formato `CONV kh kw cin cout bias`, `POOL` o `FC in out bias`.

**Uscita:**

* $L$ righe nel formato `Camada i: P`.
* Ultima riga: `Total: T`.

#### 📌 Esempi

| Ingresso                                                                            | Uscita                                                                                                         | Osservazione                                                                                                        |
| ----------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------ |
| 3<br>CONV 3 3 1 8 1<br>POOL<br>FC 1352 10 1                                         | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 13530<br>Total: 13610                                                 | Rete semplice con una convoluzione, *pooling* e layer di classificazione.                                          |
| 5<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 400 10 1               | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 4010<br>Total: 5258                  | Piccola CNN con due convoluzioni, due *pooling* e un layer completamente connesso.                                 |
| 6<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 256 32 1<br>FC 32 10 1 | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 8224<br>Camada 6: 330<br>Total: 9802 | Piccola CNN con due convoluzioni, *pooling* intermedio e due layer completamente connessi per la classificazione.  |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0903" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Conteggio dei Parametri</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 condivisione dei pesi</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>CONV</b> Blocco blu
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:50%;display:inline-block;"></span>
        <b>POOL</b> Cilindro verde
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;display:inline-block;transform:rotate(45deg);"></span>
        <b>FC</b> Rombo arancione
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;display:inline-block;"></span>
        <b>BATCH</b> Pila rossa
      </span>
      <span style="display:flex;align-items:center;gap:3px;color:#666;">
        🖱️ Trascina per spostare i livelli
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">H×W</label>
            <span id="ep0903_hw_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">32×32</span>
          </div>
          <input id="ep0903_hw" style="width:100%;accent-color:#2980b9;height:4px;" max="64" min="8" step="2" type="range" value="32">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Canali</label>
            <span id="ep0903_cin_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">1</span>
          </div>
          <input id="ep0903_cin" style="width:100%;accent-color:#2980b9;height:4px;" max="3" min="1" step="1" type="range" value="1">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Batch</label>
            <span id="ep0903_batch_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">4</span>
          </div>
          <input id="ep0903_batch" style="width:100%;accent-color:#2980b9;height:4px;" max="16" min="1" step="1" type="range" value="4">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Livelli</label>
            <span id="ep0903_nlayers_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">3</span>
          </div>
          <input id="ep0903_nlayers" style="width:100%;accent-color:#2980b9;height:4px;" max="6" min="1" step="1" type="range" value="3">
        </div>
      </div>
      
      <!-- Configuração das camadas compacta -->
      <div id="ep0903_layers_config" style="margin-bottom:8px;display:flex;flex-wrap:wrap;gap:6px;">
        <!-- Gerado dinamicamente -->
      </div>
      
      <div style="display:flex;gap:12px;align-items:center;font-size:10px;">
        <label style="display:flex;align-items:center;gap:4px;cursor:pointer;">
          <input id="ep0903_bias" type="checkbox" checked style="accent-color:#2980b9;width:14px;height:14px;">
          <span style="font-weight:bold;color:#2980b9;">Usa bias</span>
        </label>
      </div>
    </div>
    
    <!-- Visualização 3D -->
    <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;margin-bottom:12px;min-height:400px;">
      <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">
        🧠 Visualizzazione 3D
      </div>
      
      <div style="position:absolute;top:8px;right:8px;display:flex;gap:4px;z-index:10;">
        <button id="ep0903_pause_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          ⏸️ Pausa
        </button>
        <button id="ep0903_reset_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          🔄 Ripristina
        </button>
        <button id="ep0903_auto_layout_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          📐 Automatico
        </button>
      </div>
      
      <canvas id="ep0903_canvas" style="width:100%;height:340px;display:block;cursor:grab;"></canvas>
      
      <div style="position:absolute;bottom:6px;right:8px;color:white;font-size:9px;background:rgba(0,0,0,0.5);padding:3px 8px;border-radius:14px;">
        🖱️ Trascina livelli | Scroll zoom | P pausa
      </div>
    </div>
    
    <!-- Resumo compacto -->
    <div id="ep0903_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var hwEl = root.querySelector('#ep0903_hw'), hwvEl = root.querySelector('#ep0903_hw_v');
    var cinEl = root.querySelector('#ep0903_cin'), cinvEl = root.querySelector('#ep0903_cin_v');
    var batchEl = root.querySelector('#ep0903_batch'), batchvEl = root.querySelector('#ep0903_batch_v');
    var nlayersEl = root.querySelector('#ep0903_nlayers'), nlayersvEl = root.querySelector('#ep0903_nlayers_v');
    var layersConfigEl = root.querySelector('#ep0903_layers_config');
    var biasEl = root.querySelector('#ep0903_bias');
    var summaryEl = root.querySelector('#ep0903_summary');
    var canvas = root.querySelector('#ep0903_canvas');
    var ctx = canvas.getContext('2d');
    var pauseBtn = root.querySelector('#ep0903_pause_btn');
    var resetBtn = root.querySelector('#ep0903_reset_btn');
    var autoLayoutBtn = root.querySelector('#ep0903_auto_layout_btn');
    
    // Estado da visualização
    var rotationX = -0.3;
    var rotationY = 0.5;
    var zoom = 1;
    var isDragging = false;
    var isDraggingLayer = false;
    var selectedLayer = null;
    var lastX = 0;
    var lastY = 0;
    var autoRotate = true;
    var isPaused = false;
    var lastInteractionTime = Date.now();
    var animationId = null;
    var time = 0;
    
    // Posições das camadas
    var layerPositions = [];
    var batchPosition = { x: -6, y: -0.5, z: 0 };
    
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', resizeCanvas);
    
    function autoLayout() {
      var nlayers = parseInt(nlayersEl.value);
      var spacing = 4;
      var startX = -((nlayers) * spacing) / 2;
      
      batchPosition = { x: startX - spacing / 2, y: -0.5, z: 0 };
      
      layerPositions = [];
      for (var i = 0; i < nlayers; i++) {
        layerPositions.push({
          x: startX + (i + 0.5) * spacing,
          y: i * 1.5,
          z: 0
        });
      }
    }
    
    function togglePause() {
      isPaused = !isPaused;
      if (isPaused) {
        pauseBtn.textContent = '▶️';
        pauseBtn.style.background = 'rgba(76, 175, 80, 0.4)';
        autoRotate = false;
      } else {
        pauseBtn.textContent = '⏸️';
        pauseBtn.style.background = 'rgba(255,255,255,0.2)';
        autoRotate = true;
        lastInteractionTime = Date.now();
      }
    }
    
    function resetView() {
      rotationX = -0.3;
      rotationY = 0.5;
      zoom = 1;
      isPaused = false;
      autoRotate = true;
      pauseBtn.textContent = '⏸️';
      pauseBtn.style.background = 'rgba(255,255,255,0.2)';
      lastInteractionTime = Date.now();
      autoLayout();
    }
    
    pauseBtn.addEventListener('click', togglePause);
    resetBtn.addEventListener('click', resetView);
    autoLayoutBtn.addEventListener('click', autoLayout);
    
    document.addEventListener('keydown', function(e) {
      if (e.key === 'p' || e.key === 'P') togglePause();
      if (e.key === 'r' || e.key === 'R') resetView();
      if (e.key === 'a' || e.key === 'A') autoLayout();
    });
    
    function findLayerAt(mouseX, mouseY, layers) {
      var minDist = Infinity;
      var foundLayer = null;
      
      var batchProj = project(batchPosition);
      var batchDist = Math.sqrt(Math.pow(batchProj.x - mouseX, 2) + Math.pow(batchProj.y - mouseY, 2));
      if (batchDist < 50) {
        minDist = batchDist;
        foundLayer = { type: 'batch', index: -1 };
      }
      
      for (var i = 0; i < layerPositions.length && i < layers.length; i++) {
        var proj = project(layerPositions[i]);
        var dist = Math.sqrt(Math.pow(proj.x - mouseX, 2) + Math.pow(proj.y - mouseY, 2));
        
        if (dist < minDist && dist < 60) {
          minDist = dist;
          foundLayer = { type: 'layer', index: i };
        }
      }
      
      return foundLayer;
    }
    
    canvas.addEventListener('mousedown', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      var layers = getCurrentLayersInfo();
      var clickedLayer = findLayerAt(mouseX, mouseY, layers);
      
      if (clickedLayer) {
        isDraggingLayer = true;
        selectedLayer = clickedLayer;
        canvas.style.cursor = 'grabbing';
      } else {
        isDragging = true;
        canvas.style.cursor = 'grabbing';
      }
      
      autoRotate = false;
      lastX = e.clientX;
      lastY = e.clientY;
      lastInteractionTime = Date.now();
    });
    
    canvas.addEventListener('mousemove', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      if (isDraggingLayer && selectedLayer) {
        var deltaX = (e.clientX - lastX) * 0.05;
        var deltaY = -(e.clientY - lastY) * 0.05;
        
        if (selectedLayer.type === 'batch') {
          batchPosition.x += deltaX;
          batchPosition.y += deltaY;
        } else if (selectedLayer.type === 'layer') {
          layerPositions[selectedLayer.index].x += deltaX;
          layerPositions[selectedLayer.index].y += deltaY;
        }
        
        lastX = e.clientX;
        lastY = e.clientY;
      } else if (isDragging) {
        var deltaX = e.clientX - lastX;
        var deltaY = e.clientY - lastY;
        rotationY += deltaX * 0.01;
        rotationX += deltaY * 0.01;
        rotationX = Math.max(-1.5, Math.min(1.5, rotationX));
        lastX = e.clientX;
        lastY = e.clientY;
      }
      
      if (!isDragging && !isDraggingLayer) {
        var layers = getCurrentLayersInfo();
        var hoveredLayer = findLayerAt(mouseX, mouseY, layers);
        canvas.style.cursor = hoveredLayer ? 'pointer' : 'grab';
      }
    });
    
    canvas.addEventListener('mouseup', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    canvas.addEventListener('mouseleave', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
    });
    
    canvas.addEventListener('wheel', function(e) {
      e.preventDefault();
      zoom *= (1 + e.deltaY * 0.001);
      zoom = Math.max(0.5, Math.min(2, zoom));
      lastInteractionTime = Date.now();
      autoRotate = false;
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    function rotateX(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x, y: point.y * cos - point.z * sin, z: point.y * sin + point.z * cos };
    }
    
    function rotateY(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x * cos - point.z * sin, y: point.y, z: point.x * sin + point.z * cos };
    }
    
    function project(point) {
      var rotated = rotateX(point, rotationX);
      rotated = rotateY(rotated, rotationY);
      var scale = zoom * 30;
      return { x: canvas.width / 2 + rotated.x * scale, y: canvas.height / 2 - rotated.y * scale, z: rotated.z };
    }
    
    function shadeColor(color, percent) {
      var num = parseInt(color.replace('#', ''), 16);
      var amt = Math.round(2.55 * percent);
      var R = (num >> 16) + amt;
      var G = (num >> 8 & 0x00FF) + amt;
      var B = (num & 0x0000FF) + amt;
      return '#' + (0x1000000 + (R < 255 ? R < 1 ? 0 : R : 255) * 0x10000 + (G < 255 ? G < 1 ? 0 : G : 255) * 0x100 + (B < 255 ? B < 1 ? 0 : B : 255)).toString(16).slice(1);
    }
    
    function draw3DBox(x, y, z, width, height, depth, color, opacity, label, shape) {
      shape = shape || 'box';
      if (shape === 'cylinder') { draw3DCylinder(x, y, z, width, height, depth, color, opacity, label); return; }
      if (shape === 'diamond') { draw3DDiamond(x, y, z, width, height, depth, color, opacity, label); return; }
      
      var vertices = [
        {x: x - width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y + height/2, z: z + depth/2},
        {x: x - width/2, y: y + height/2, z: z + depth/2}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2, 3], color: shadeColor(color, -20)},
        {vertices: [4, 5, 6, 7], color: shadeColor(color, 20)},
        {vertices: [0, 1, 5, 4], color: shadeColor(color, -40)},
        {vertices: [2, 3, 7, 6], color: shadeColor(color, 40)},
        {vertices: [1, 2, 6, 5], color: shadeColor(color, -10)},
        {vertices: [0, 3, 7, 4], color: shadeColor(color, 10)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DCylinder(x, y, z, width, height, depth, color, opacity, label) {
      var segments = 12;
      var topVertices = [];
      var bottomVertices = [];
      
      for (var i = 0; i < segments; i++) {
        var angle = (i / segments) * Math.PI * 2;
        var cx = x + Math.cos(angle) * width / 2;
        var cz = z + Math.sin(angle) * depth / 2;
        topVertices.push({x: cx, y: y + height/2, z: cz});
        bottomVertices.push({x: cx, y: y - height/2, z: cz});
      }
      
      var projectedTop = topVertices.map(function(v) { return project(v); });
      var projectedBottom = bottomVertices.map(function(v) { return project(v); });
      
      for (var i = 0; i < segments; i++) {
        var next = (i + 1) % segments;
        ctx.beginPath();
        ctx.moveTo(projectedTop[i].x, projectedTop[i].y);
        ctx.lineTo(projectedTop[next].x, projectedTop[next].y);
        ctx.lineTo(projectedBottom[next].x, projectedBottom[next].y);
        ctx.lineTo(projectedBottom[i].x, projectedBottom[i].y);
        ctx.closePath();
        ctx.fillStyle = shadeColor(color, (i % 2 === 0) ? -10 : 10);
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      }
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DDiamond(x, y, z, width, height, depth, color, opacity, label) {
      var vertices = [
        {x: x, y: y + height/2, z: z},
        {x: x + width/2, y: y, z: z},
        {x: x, y: y, z: z + depth/2},
        {x: x - width/2, y: y, z: z},
        {x: x, y: y, z: z - depth/2},
        {x: x, y: y - height/2, z: z}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2], color: shadeColor(color, -20)},
        {vertices: [0, 2, 3], color: shadeColor(color, 20)},
        {vertices: [0, 3, 4], color: shadeColor(color, -10)},
        {vertices: [0, 4, 1], color: shadeColor(color, 10)},
        {vertices: [5, 1, 2], color: shadeColor(color, -30)},
        {vertices: [5, 2, 3], color: shadeColor(color, 30)},
        {vertices: [5, 3, 4], color: shadeColor(color, -20)},
        {vertices: [5, 4, 1], color: shadeColor(color, 20)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function drawImageBatch(x, y, z, width, height, numImages, color) {
      var imageDepth = 0.3;
      var gap = 0.1;
      var totalDepth = numImages * (imageDepth + gap);
      var startZ = z - totalDepth / 2;
      
      for (var i = 0; i < numImages; i++) {
        var imageZ = startZ + i * (imageDepth + gap);
        var alpha = 0.3 + (i / numImages) * 0.5;
        draw3DBox(x, y, imageZ, width, height, imageDepth, color, alpha, null, 'box');
      }
    }
    
    function drawConnection(x1, y1, z1, x2, y2, z2, animated) {
      var start = project({x: x1, y: y1, z: z1});
      var end = project({x: x2, y: y2, z: z2});
      var midX = (start.x + end.x) / 2;
      var midY = Math.min(start.y, end.y) - 20;
      
      if (animated && !isPaused) {
        var pulse = Math.sin(time * 0.002) * 0.5 + 0.5;
        ctx.strokeStyle = 'rgba(255, 255, 255, ' + (0.3 + pulse * 0.3) + ')';
        ctx.lineWidth = 1.5 + pulse;
      } else {
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1.5;
      }
      
      ctx.setLineDash([4, 4]);
      ctx.beginPath();
      ctx.moveTo(start.x, start.y);
      ctx.quadraticCurveTo(midX, midY, end.x, end.y);
      ctx.stroke();
      ctx.setLineDash([]);
    }
    
    function getCurrentLayersInfo() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var layersInfo = [];
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect ? typeSelect.value : 'CONV';
        var inputStr = '';
        var outputStr = '';
        
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          // Se a camada anterior era FC, usa a saída dela
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            // Se veio de CONV/POOL, faz flatten
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          outputStr = fcout;
          currentFCInput = fcout;
          // Após FC, não há mais dimensões espaciais
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
      }
      
      return layersInfo;
    }
    
    function render3D(layers, batchSize) {
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      if (!isPaused) time += 16;
      if (autoRotate && !isDragging && !isPaused && Date.now() - lastInteractionTime > 3000) rotationY += 0.005;
      
      // Grid
      ctx.strokeStyle = 'rgba(255, 255, 255, 0.08)';
      ctx.lineWidth = 0.5;
      for (var i = -10; i <= 10; i++) {
        var start = project({x: i * 2, y: -2, z: -10 * 2});
        var end = project({x: i * 2, y: -2, z: 10 * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
        start = project({x: -10 * 2, y: -2, z: i * 2});
        end = project({x: 10 * 2, y: -2, z: i * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
      }
      
      var colors = ['#4a90e2', '#50e3c2', '#f5a623', '#d0021b', '#8b572a', '#9013fe'];
      
      // Batch (apenas se a primeira camada for CONV ou POOL)
      if (layers.length > 0 && (layers[0].type === 'CONV' || layers[0].type === 'POOL')) {
        var inputWidth = Math.max(1, Math.min(4, layers[0].h / 8));
        var inputHeight = Math.max(1, Math.min(4, layers[0].w / 8));
        var labelPos = project({x: batchPosition.x, y: batchPosition.y + inputHeight/2 + 0.7, z: batchPosition.z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 9px Arial';
        ctx.textAlign = 'center';
        ctx.fillText('BATCH: ' + batchSize, labelPos.x, labelPos.y);
        drawImageBatch(batchPosition.x, batchPosition.y, batchPosition.z, inputWidth, inputHeight, batchSize, '#ff6b6b');
        
        // Conexão batch -> primeira camada
        drawConnection(batchPosition.x + inputWidth/2, batchPosition.y, batchPosition.z, layerPositions[0].x - Math.max(1, Math.min(4, layers[0].h / 8))/2, layerPositions[0].y, layerPositions[0].z, true);
      }
      
      // Conexões entre camadas
      for (var i = 0; i < layers.length - 1 && i < layerPositions.length - 1; i++) {
        drawConnection(layerPositions[i].x + 1, layerPositions[i].y, layerPositions[i].z, layerPositions[i + 1].x - 1, layerPositions[i + 1].y, layerPositions[i + 1].z, true);
      }
      
      // Camadas
      for (var i = 0; i < layers.length && i < layerPositions.length; i++) {
        var layer = layers[i];
        var pos = layerPositions[i];
        
        var color = colors[i % colors.length];
        var label = '';
        
        if (layer.type === 'CONV') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'box');
        } else if (layer.type === 'POOL') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'cylinder');
        } else if (layer.type === 'FC') {
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, 1, 1, 1, color, 0.7, label, 'diamond');
        }
      }
      
      animationId = requestAnimationFrame(function() { render3D(layers, batchSize); });
    }
    
    function generateLayerConfig() {
      var nlayers = parseInt(nlayersEl.value);
      var html = '';
      
      for (var i = 0; i < nlayers; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:6px 8px;display:flex;gap:6px;align-items:center;flex-wrap:wrap;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">' + (i+1) + ':</span>';
        html += '<select id="ep0903_type_' + i + '" style="padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">';
        html += '<option value="CONV"' + (i < 2 ? ' selected' : '') + '>CONV</option>';
        html += '<option value="POOL">POOL</option>';
        html += '<option value="FC"' + (i >= 2 ? ' selected' : '') + '>FC</option>';
        html += '</select>';
        html += '<div id="ep0903_params_' + i + '" style="display:flex;gap:3px;flex-wrap:wrap;"></div>';
        html += '</div>';
      }
      
      layersConfigEl.innerHTML = html;
      
      for (var i = 0; i < nlayers; i++) {
        (function(index) {
          var typeSelect = root.querySelector('#ep0903_type_' + index);
          typeSelect.addEventListener('change', function() {
            updateLayerParams(index);
            render();
          });
          updateLayerParams(index);
        })(i);
      }
      
      autoLayout();
    }
    
    function updateLayerParams(index) {
      var typeSelect = root.querySelector('#ep0903_type_' + index);
      var paramsDiv = root.querySelector('#ep0903_params_' + index);
      var type = typeSelect.value;
      
      if (type === 'CONV') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_kh_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel h">' +
          '<span style="font-size:8px;">×</span>' +
          '<input type="number" id="ep0903_kw_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel w">' +
          '<input type="number" id="ep0903_cout_' + index + '" value="' + (index === 0 ? '8' : '16') + '" min="1" max="64" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="filtri">';
      } else if (type === 'POOL') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_pool_size_' + index + '" value="2" min="2" max="4" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="pool size">';
      } else if (type === 'FC') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_fcout_' + index + '" value="10" min="1" max="100" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="uscite">';
      }
      
      var inputs = paramsDiv.querySelectorAll('input');
      inputs.forEach(function(input) {
        input.addEventListener('input', render);
        input.addEventListener('change', render);
      });
    }
    
    function render() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var batchSize = parseInt(batchEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var bias = biasEl.checked ? 1 : 0;
      
      hwvEl.textContent = hw + '×' + hw;
      cinvEl.textContent = cin;
      batchvEl.textContent = batchSize;
      nlayersvEl.textContent = nlayers;
      
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var totalParams = 0;
      var layersInfo = [];
      var summaryHTML = '';
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect.value;
        var params = 0;
        var inputStr = '';
        var outputStr = '';
        
        // Determinar entrada
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        // Processar camada
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          params = kh * kw * currentCin * cout + cout * bias;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          params = 0;
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          var fcin = parseInt(inputStr);
          params = fcin * fcout + fcout * bias;
          outputStr = fcout;
          currentFCInput = fcout;
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        totalParams += params;
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          params: params,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
        
        summaryHTML += '<span style="color:' + (type === 'CONV' ? '#2980b9' : type === 'POOL' ? '#666' : '#009933') + ';font-weight:bold;">' + (i+1) + ' (' + type + '):</span> ';
        summaryHTML += inputStr + ' → ' + outputStr;
        summaryHTML += ' [' + params.toLocaleString('pt-BR') + ']<br>';
      }
      
      summaryHTML += '<b>Total: ' + totalParams.toLocaleString('pt-BR') + ' parâmetros</b>';
      summaryEl.innerHTML = summaryHTML;
      
      if (animationId) cancelAnimationFrame(animationId);
      render3D(layersInfo, batchSize);
    }
    
    // Inicializar
    autoLayout();
    generateLayerConfig();
    render();
    
    // Event listeners
    hwEl.addEventListener('input', render);
    cinEl.addEventListener('input', render);
    batchEl.addEventListener('input', render);
    nlayersEl.addEventListener('input', function() { generateLayerConfig(); render(); });
    biasEl.addEventListener('change', render);
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0903');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.3:** Simulatore EP09_03: Conteggio dei Parametri — Convoluzione vs. Strato Totalmente Connesso


<figure id="fig-09-sim-ep0903">
  <img src="imagens/fig-09-sim-ep0903.png" alt=" Simulatore EP09_03: Conteggio dei Parametri — Convoluzione vs. Strato Totalmente Connesso " style="max-width:80%" />
  <figcaption><strong>Figura 9.3:</strong>  Simulatore EP09_03: Conteggio dei Parametri — Convoluzione vs. Strato Totalmente Connesso </figcaption>
</figure>

In [ ]:
%%writefile EP09_03.py
# Codice Python

In [ ]:
TestSuite("EP09_03.py").run()

### EP09_04 🟡 Intersezione su Unione (IoU) e Soppressione dei Non-Massimi (NMS)

I modelli di rilevamento degli oggetti possono produrre **diverse bounding box candidate** per lo stesso oggetto, con posizioni e punteggi di confidenza differenti. La fase di post-elaborazione responsabile dell'eliminazione di queste rilevazioni ridondanti è la **Soppressione dei Non-Massimi (NMS)**, la cui operazione fondamentale utilizza la metrica di **Intersezione su Unione (IoU)**.

La NMS utilizza questa misura per decidere quali box devono essere mantenute. In generale, la box con la confidenza più alta viene selezionata per prima; successivamente, le box che presentano un'IoU superiore a una certa soglia con la box selezionata sono considerate ridondanti e vengono rimosse. Il processo viene ripetuto finché non rimangono box candidate.

In questo esercizio, dovrai implementare l'algoritmo NMS da zero, calcolando l'IoU tra le box e applicando successivamente il criterio di selezione e soppressione per produrre l'insieme finale di rilevazioni.

#### 📋 Linee Guida di Implementazione

1. **Input:** Leggere l'intero $N$ (numero di box candidate) e la soglia reale $\tau$ (soglia IoU per la soppressione), sulla stessa riga.

2. **Box:** Leggere $N$ righe, ciascuna con cinque valori reali:

   `x1 y1 x2 y2 score`

   dove $(x_1,y_1)$ rappresenta l'angolo superiore sinistro, $(x_2,y_2)$ l'angolo inferiore destro e `score` il punteggio di confidenza.

3. **Intersezione su Unione:** Per due box $A$ e $B$,

   $$
   IoU(A,B)=
   \frac{\operatorname{Area}(A\cap B)}
   {\operatorname{Area}(A\cup B)}.
   $$

   L'area di intersezione deve essere calcolata dalla sovrapposizione degli intervalli in $x$ e $y$. Se non c'è sovrapposizione, l'area di intersezione è zero.

4. **Algoritmo greedy di NMS:**

   a. Ordina le box per `score` decrescente. In caso di parità, mantieni l'ordine originale di lettura.

   b. Seleziona la box con il punteggio più alto tra le box rimanenti e aggiungila all'insieme di output.

   c. Calcola l'IoU tra la box selezionata e **tutte le box ancora rimanenti**. Sopprimi le box per cui

   $$
   \text{IoU} > \tau.
   $$

   d. Ripeti i passaggi (b) e (c) finché non rimangono box.

5. **Output:** Per ogni box mantenuta, nell'ordine in cui è stata selezionata, stampa il suo indice originale (posizione di lettura, a partire da $0$) e il suo `score`, formattato con 4 cifre decimali. Alla fine, stampa:

   `Totale mantenute: X`

#### 📌 Vincoli Computazionali

* **Soppressione stretta:** solo le box con $\text{IoU} > \tau$ vengono soppresse. Le box con $\text{IoU}=\tau$ vengono mantenute.
* **Indici originali:** l'output fa riferimento alla posizione in cui ogni box è stata letta nell'input (a partire da $0$), non alla sua posizione dopo l'ordinamento.
* **Ordinamento stabile:** in caso di `score` uguali, deve essere preservato l'ordine originale di lettura.
* **Rettangoli allineati agli assi:** tutte le box sono specificate da due angoli, con $x_1 < x_2$ e $y_1 < y_2$ garantiti nell'input.
* **Coordinate e punteggi:** i valori reali possono essere positivi o negativi, secondo i limiti definiti dall'input, ma le dimensioni delle box sono sempre positive.

#### 🧠 Fondamento Teorico

| Elemento                | Ruolo nella post-elaborazione della rilevazione                                                                                          |
| ----------------------- | ---------------------------------------------------------------------------------------------------------------------------------------- |
| IoU                     | Quantifica la sovrapposizione spaziale tra due box; $\text{IoU}=1$ per box identiche e $\text{IoU}=0$ per box senza sovrapposizione |
| Ordinamento per confidenza | Fa sì che la box con `score` più alto venga analizzata per prima                                                                        |
| Soglia $\tau$           | Definisce la quantità di sovrapposizione necessaria affinché una box sia considerata ridondante                                           |
| Soppressione            | Rimuove le box che presentano una grande sovrapposizione con una box già selezionata                                                     |
| Box distanti            | Hanno IoU vicina a zero e, in generale, non vengono soppresse da questa regola                                                           |

#### 🧩 Metodi di `morph.py` che possono aiutare

* `mm.IoU(boxA, boxB)` — calcola la metrica IoU, ma si aspetta le box nel formato $(x,y,w,h)$, cioè angolo superiore sinistro, larghezza e altezza. L'input di questo esercizio utilizza il formato $(x_1,y_1,x_2,y_2)$. La conversione è diretta:

  $$
  w=x_2-x_1,\qquad h=y_2-y_1.
  $$

  L'uso di questa funzione è facoltativo. L'obiettivo principale dell'esercizio è implementare correttamente il processo di selezione e soppressione della NMS.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: intero $N$ e reale $\tau$.
* Prossime $N$ righe: $x_1\ y_1\ x_2\ y_2\ \text{score}$.

**Output:**

* Una riga per box mantenuta, nell'ordine di selezione: `indice score`.
* Ultima riga: `Totale mantenute: X`.

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0904" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: IoU e Soppressione Non-Massimale</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 NMS</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Casella selezionata</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Casella mantenuta</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Casella soppressa</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Casella candidata</b>
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Numero di caselle</label>
            <span id="ep0904_n_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5</span>
          </div>
          <input id="ep0904_n" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="2" step="1" type="range" value="5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Soglia τ (IoU)</label>
            <span id="ep0904_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0904_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Esempio</label>
            <span id="ep0904_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Predefinito</span>
          </div>
          <select id="ep0904_example" style="width:100%;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">
            <option value="padrao">Esempio predefinito</option>
            <option value="agrupado">Caselle raggruppate</option>
            <option value="disperso">Caselle sparse</option>
            <option value="aninhado">Caselle annidate</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0904_run_btn" style="background:#2980b9;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;transition:all 0.3s;">
            ▶️ Esegui NMS
          </button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0904_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:8px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:300px;">
        <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">
          🎯 Visualizzazione delle caselle
        </div>
        <canvas id="ep0904_canvas" style="width:100%;height:280px;display:block;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:310px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📋 Procedura passo passo della NMS
        </div>
        <div id="ep0904_steps" style="font-family:monospace;font-size:10px;line-height:1.6;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo -->
    <div id="ep0904_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var nEl = root.querySelector('#ep0904_n'), nvEl = root.querySelector('#ep0904_n_v');
    var tauEl = root.querySelector('#ep0904_tau'), tauvEl = root.querySelector('#ep0904_tau_v');
    var exampleEl = root.querySelector('#ep0904_example');
    var boxesConfigEl = root.querySelector('#ep0904_boxes_config');
    var canvas = root.querySelector('#ep0904_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0904_steps');
    var summaryEl = root.querySelector('#ep0904_summary');
    var runBtn = root.querySelector('#ep0904_run_btn');
    
    // Estado
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    
    // Exemplos pré-definidos
    var examples = {
      padrao: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 30, y1: 15, x2: 70, y2: 55, score: 0.7},
        {x1: 80, y1: 80, x2: 120, y2: 120, score: 0.6},
        {x1: 85, y1: 85, x2: 125, y2: 125, score: 0.5}
      ],
      agrupado: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 15, y1: 15, x2: 55, y2: 55, score: 0.85},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 25, y1: 25, x2: 65, y2: 65, score: 0.75},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7}
      ],
      disperso: [
        {x1: 10, y1: 10, x2: 40, y2: 40, score: 0.9},
        {x1: 80, y1: 10, x2: 110, y2: 40, score: 0.8},
        {x1: 10, y1: 80, x2: 40, y2: 110, score: 0.7},
        {x1: 80, y1: 80, x2: 110, y2: 110, score: 0.6},
        {x1: 45, y1: 45, x2: 75, y2: 75, score: 0.5}
      ],
      aninhado: [
        {x1: 10, y1: 10, x2: 90, y2: 90, score: 0.9},
        {x1: 20, y1: 20, x2: 80, y2: 80, score: 0.8},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7},
        {x1: 40, y1: 40, x2: 60, y2: 60, score: 0.6},
        {x1: 45, y1: 45, x2: 55, y2: 55, score: 0.5}
      ]
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', function() {
      resizeCanvas();
      render();
    });
    
    // Carregar exemplo
    function loadExample(name) {
      boxes = JSON.parse(JSON.stringify(examples[name]));
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas carregadas: ' + boxes.length + '. Clique em "Executar NMS".';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      for (var i = 0; i < boxes.length; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">#' + i + ':</span>';
        html += '<input type="number" id="ep0904_x1_' + i + '" value="' + boxes[i].x1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x1">';
        html += '<input type="number" id="ep0904_y1_' + i + '" value="' + boxes[i].y1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y1">';
        html += '<input type="number" id="ep0904_x2_' + i + '" value="' + boxes[i].x2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x2">';
        html += '<input type="number" id="ep0904_y2_' + i + '" value="' + boxes[i].y2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y2">';
        html += '<input type="number" id="ep0904_score_' + i + '" value="' + boxes[i].score + '" step="0.05" min="0" max="1" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="punteggio">';
        html += '</div>';
      }
      boxesConfigEl.innerHTML = html;
      
      // Adicionar event listeners
      for (var i = 0; i < boxes.length; i++) {
        (function(index) {
          ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
            var input = root.querySelector('#ep0904_' + field + '_' + index);
            if (input) {
              input.addEventListener('input', function() {
                boxes[index][field] = parseFloat(input.value) || 0;
                selectedBoxes = [];
                suppressedBoxes = [];
                render();
              });
            }
          });
        })(i);
      }
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar NMS
    function runNMS() {
      var tau = parseFloat(tauEl.value);
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      // Ordenar por score decrescente (estável)
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) {
          return b.box.score - a.box.score;
        }
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push({
          type: 'select',
          box: selected,
          remaining: remaining.slice()
        });
        
        var newRemaining = [];
        var suppressed = [];
        
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressed.push({ box: remaining[i], iou: iou });
            suppressedBoxes.push(remaining[i]);
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        
        if (suppressed.length > 0) {
          steps.push({
            type: 'suppress',
            box: selected,
            suppressed: suppressed,
            remaining: newRemaining.slice()
          });
        }
        
        remaining = newRemaining;
      }
      
      return steps;
    }
    
    // Renderizar visualização
    function render() {
      var tau = parseFloat(tauEl.value);
      nvEl.textContent = boxes.length;
      tauvEl.textContent = tau.toFixed(2);
      
      // Limpar canvas
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      
      // Desenhar todas as caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        
        var color = '#f5a623'; // candidata
        if (isSelected) color = '#4a90e2'; // selecionada
        if (isSuppressed) color = '#ff6b6b'; // suprimida
        
        drawBox(box, color, index);
      });
      
      // Atualizar resumo
      var summaryHTML = '';
      if (selectedBoxes.length > 0) {
        summaryHTML += '<b>Caixas selecionadas (em ordem):</b><br>';
        selectedBoxes.forEach(function(s) {
          summaryHTML += '#' + s.originalIndex + ' (score: ' + s.box.score.toFixed(4) + ')<br>';
        });
        summaryHTML += '<b>Total mantidas: ' + selectedBoxes.length + '</b>';
        summaryEl.innerHTML = summaryHTML;
      }
    }
    
    // Desenhar caixa
    function drawBox(box, color, index) {
      var scale = 2.0;
      var offsetX = 30;
      var offsetY = 30;
      
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      // Desenhar caixa
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      // Preenchimento translúcido
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      // Label
      ctx.fillStyle = color;
      ctx.font = 'bold 12px Arial';
      ctx.textAlign = 'center';
      ctx.fillText('#' + index, x + w/2, y - 5);
      
      // Score
      ctx.fillStyle = '#666';
      ctx.font = '10px Arial';
      ctx.fillText('punteggio: ' + box.score.toFixed(2), x + w/2, y + h/2);
    }
    
    // Mostrar passo a passo
    function showSteps(steps) {
      var html = '';
      
      steps.forEach(function(step, index) {
        if (step.type === 'select') {
          html += '<div style="color:#4a90e2;font-weight:bold;margin-top:4px;">';
          html += 'Passo ' + (index + 1) + ': Selecionar caixa #' + step.box.originalIndex;
          html += ' (score: ' + step.box.box.score.toFixed(4) + ')';
          html += '</div>';
        } else if (step.type === 'suppress') {
          html += '<div style="color:#ff6b6b;margin-left:10px;">';
          html += '↳ Suprimir: ';
          step.suppressed.forEach(function(s, i) {
            if (i > 0) html += ', ';
            html += '#' + s.box.originalIndex;
            html += ' (IoU: ' + s.iou.toFixed(3) + ')';
          });
          html += '</div>';
        }
      });
      
      if (selectedBoxes.length > 0) {
        html += '<div style="color:#50e3c2;font-weight:bold;margin-top:8px;">';
        html += '✓ Resultado: ' + selectedBoxes.length + ' caixa(s) mantida(s)';
        html += '</div>';
      }
      
      stepsEl.innerHTML = html;
    }
    
    // Executar NMS
    function executeNMS() {
      var steps = runNMS();
      showSteps(steps);
      render();
      
      // Animação do botão
      runBtn.textContent = '✓ Executado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Executar NMS';
        runBtn.style.background = '#2980b9';
      }, 1000);
    }
    
    // Event listeners
    runBtn.addEventListener('click', executeNMS);
    
    nEl.addEventListener('input', function() {
      var n = parseInt(nEl.value);
      var currentN = boxes.length;
      
      if (n > currentN) {
        for (var i = currentN; i < n; i++) {
          boxes.push({
            x1: 10 + i * 5,
            y1: 10 + i * 5,
            x2: 50 + i * 5,
            y2: 50 + i * 5,
            score: 0.9 - i * 0.1
          });
        }
      } else if (n < currentN) {
        boxes = boxes.slice(0, n);
      }
      
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas atualizadas: ' + boxes.length + '. Clique em "Executar NMS".';
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Limiar atualizado. Clique em "Executar NMS".';
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0904');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.4:** Simulatore EP09_04: IoU e Soppressione Non-Massima (NMS)


<figure id="fig-09-sim-ep0904">
  <img src="imagens/fig-09-sim-ep0904.png" alt=" Simulatore EP09_04: IoU e Soppressione Non-Massima (NMS) " style="max-width:80%" />
  <figcaption><strong>Figura 9.4:</strong>  Simulatore EP09_04: IoU e Soppressione Non-Massima (NMS) </figcaption>
</figure>

In [ ]:
%%writefile EP09_04.py
# Codice Python


In [ ]:
TestSuite("EP09_04.py").run()

### EP09_05 🟠 Valutazione della Segmentazione: IoU e Dice Pixel per Pixel

Il Blocco 2 della sezione "Segmentazione Semantica con Architettura U-Net" definisce, in poche righe, la funzione `iou_mascaras`, utilizzata per misurare la qualità della baseline morfologica classica (smoothing + Otsu + apertura) e, più avanti, della stessa U-Net addestrata. Diversamente dall'IoU dell'EP09_04 — calcolato su **bounding box** (regioni rettangolari descritte da quattro numeri) —, l'IoU di segmentazione è calcolato **pixel per pixel**: ogni posizione dell'immagine viene confrontata individualmente tra la maschera predetta e la maschera di riferimento.

Ti è stato affidato il compito di generalizzare questa valutazione, implementando non solo l'IoU pixel per pixel, ma anche il **coefficiente di Dice**, un'altra metrica di sovrapposizione ampiamente utilizzata in segmentazione medica (inclusa nella funzione `perda_dice`, menzionata nello stesso blocco del capitolo come base della funzione di perdita utilizzata per addestrare la U-Net).

#### 📋 Linee Guida di Implementazione

1. **Input:** Leggere le dimensioni $H \times W$ delle maschere.

2. **Maschera predetta:** Leggere $H$ righe con $W$ valori interi (0 o 1) ciascuna — ad esempio, l'output di una U-Net dopo la sogliatura a $0{,}5$ sulla sigmoide, come nel Blocco 4 del capitolo.

3. **Maschera di riferimento:** Leggere altre $H$ righe con $W$ valori interi (0 o 1) ciascuna — il *ground truth*.

4. **Intersezione e unione:** Considerando ogni pixel come appartenente all'oggetto quando il suo valore è diverso da zero,
   $$
   \text{intersezione} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \wedge R_{ij}=1], \qquad
   \text{unione} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \vee R_{ij}=1].
   $$

5. **IoU pixel per pixel:**
   $$
   \text{IoU} = \frac{\text{intersezione}}{\text{unione}}.
   $$

6. **Coefficiente di Dice:**
   $$
   \text{Dice} = \frac{2 \cdot \text{intersezione}}{|P| + |R|},
   $$
   dove $|P|$ e $|R|$ sono il numero totale di pixel dell'oggetto in ciascuna maschera.

7. **Convenzione per maschere vuote:** se **entrambe** le maschere non hanno alcun pixel dell'oggetto (unione $= 0$ e $|P|+|R|=0$), considera la corrispondenza banalmente perfetta: $\text{IoU} = \text{Dice} = 1{,}0$.

8. **Output:** Due righe, `IoU: X.XXXX` e `Dice: X.XXXX`, ciascun valore con 4 cifre decimali.

#### 📌 Vincoli Computazionali

* **Qualsiasi valore non nullo conta come oggetto:** tratta i valori diversi da $0$ (non solo $1$) come appartenenti alla maschera, replicando il controllo `predita > 0` usato in `iou_mascaras` nel capitolo.
* **Stesse dimensioni:** le due maschere hanno sempre esattamente $H \times W$ elementi.
* **Convenzione del vuoto:** applica la regola del punto 7 **solo** quando entrambe le maschere sono completamente vuote; se solo una è vuota, l'intersezione è $0$ e l'IoU/Dice risultante sarà anch'esso $0$.

#### 🧠 Fondamento Teorico

| Elemento | Ruolo nella valutazione della segmentazione |
|---|---|
| IoU pixel per pixel | Generalizza la metrica dell'EP09_04 a regioni di forma arbitraria — non solo rettangoli — confrontando la maschera predetta e il riferimento posizione per posizione |
| Coefficiente di Dice | Metrica correlata all'IoU (sempre $\text{Dice} \ge \text{IoU}$), più sensibile a piccole intersezioni e ampiamente utilizzata come funzione di perdita in segmentazione (funzione `perda_dice` del capitolo) |
| Convenzione per maschere vuote | Evita la divisione per zero e riconosce che "nessun oggetto previsto, nessun oggetto reale" è, per definizione, un successo |
| Confronto classico vs. U-Net | Il capitolo usa esattamente questo tipo di metrica per giustificare, numericamente, perché la U-Net supera la baseline morfologica in scenari a basso contrasto |

#### 🧩 Metodi di `morph.py` che possono aiutare

* `mm.readImg(h, w, dtype='uint8')` — legge direttamente ogni maschera binaria $h \times w$ dall'input standard (i valori $0/1$ rientrano perfettamente nel tipo intero standard).
* La stessa funzione `iou_mascaras`, definita nel Blocco 2 della sezione U-Net del capitolo (non fa parte di `morph.py`, ma del codice del capitolo), è l'ispirazione diretta di questo esercizio — vale la pena rileggere quelle poche righe prima di programmare.
* Per un'estensione opzionale (non richiesta da questo EP), `mm.connectedComponents` o `mm.label0` (visti nel contesto dell'analisi delle componenti connesse) permetterebbero di etichettare ogni nodulo individualmente e calcolare l'IoU **per componente**, invece che sull'intera maschera.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ valori interi (0 o 1) — maschera predetta.
* Prossime $H$ righe: $W$ valori interi (0 o 1) — maschera di riferimento.

**Output:**

* Riga 1: `IoU: X.XXXX`.
* Riga 2: `Dice: X.XXXX`.

> ### 💡 Dica
>
> ##### 💡 Esempio Illustrativo
>
> Considera una maschera predetta con un quadrato $2\times2$ di pixel attivi e un riferimento spostato di una colonna, sovrapponendosi solo per metà dell'area:
>
> ```
> Predetta        Riferimento
> 0 0 0 0         0 0 0 0
> 0 1 1 0         0 0 1 1
> 0 1 1 0         0 0 1 1
> 0 0 0 0         0 0 0 0
> ```
>
> Intersezione $=2$ pixel, unione $=6$ pixel ($4+4-2$), quindi $\text{IoU}=2/6\approx0{,}3333$ e $\text{Dice}=2\cdot2/(4+4)=0{,}5000$ — nota che il Dice è sempre uguale o maggiore dell'IoU per la stessa sovrapposizione.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4 4<br>0 0 0 0<br>0 1 1 0<br>0 1 1 0<br>0 0 0 0<br>0 0 0 0<br>0 0 1 1<br>0 0 1 1<br>0 0 0 0 | IoU: 0.3333<br>Dice: 0.5000 | Maschere $4\times4$ con sovrapposizione parziale di 2 pixel. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0905" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: IoU e Dice Pixel per Pixel</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟠 Segmentazione</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Intersezione</b> (VP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Solo previsto</b> (FP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Solo riferimento</b> (FN)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f0f0f0;border:2px solid #ccc;border-radius:2px;display:inline-block;"></span>
        <b>Sfondo</b> (VN)
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:flex;gap:10px;margin-bottom:8px;flex-wrap:wrap;">
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Dimensioni</label>
            <span id="ep0905_dim_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5×5</span>
          </div>
          <input id="ep0905_dim" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="3" step="1" type="range" value="5">
        </div>
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Esempio</label>
            <span id="ep0905_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Quadrato</span>
          </div>
          <select id="ep0905_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="quadrado">Quadrato 2×2</option>
            <option value="deslocado">Spostato</option>
            <option value="perfeito">Perfetto</option>
            <option value="vazio">Maschere Vuote</option>
            <option value="parcial">Sovrapposizione Parziale</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;gap:8px;">
          <button id="ep0905_clear_btn" style="background:#666;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🗑️ Pulisci</button>
          <button id="ep0905_random_btn" style="background:#f5a623;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🎲 Casuale</button>
        </div>
      </div>
      
      <!-- Grids de máscaras -->
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:10px;">
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🔵 Maschera Prevista
          </div>
          <div id="ep0905_pred_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🟡 Maschera di Riferimento
          </div>
          <div id="ep0905_ref_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Confronto Visivo
        </div>
        <canvas id="ep0905_canvas" style="width:100%;height:220px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Métricas -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:280px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📊 Calcoli e Formule
        </div>
        <div id="ep0905_metrics" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo final -->
    <div id="ep0905_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var dimEl = root.querySelector('#ep0905_dim');
    var dimvEl = root.querySelector('#ep0905_dim_v');
    var exampleEl = root.querySelector('#ep0905_example');
    var predGridEl = root.querySelector('#ep0905_pred_grid');
    var refGridEl = root.querySelector('#ep0905_ref_grid');
    var canvas = root.querySelector('#ep0905_canvas');
    var ctx = canvas.getContext('2d');
    var metricsEl = root.querySelector('#ep0905_metrics');
    var summaryEl = root.querySelector('#ep0905_summary');
    var clearBtn = root.querySelector('#ep0905_clear_btn');
    var randomBtn = root.querySelector('#ep0905_random_btn');
    
    // Estado
    var predMask = [];
    var refMask = [];
    var H = 5;
    var W = 5;
    
    // Exemplos pré-definidos
    var examples = {
      quadrado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      deslocado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      perfeito: {
        pred: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]],
        ref: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]]
      },
      vazio: {
        pred: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      parcial: {
        pred: [[0,0,0,0,0],[0,1,1,1,0],[0,1,1,1,0],[0,1,1,1,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0]]
      }
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      predMask = example.pred.map(function(row) { return row.slice(); });
      refMask = example.ref.map(function(row) { return row.slice(); });
      H = predMask.length;
      W = predMask[0].length;
      dimEl.value = H;
      dimvEl.textContent = H + '×' + W;
      generateGrids();
      render();
    }
    
    // Gerar grids clicáveis
    function generateGrids() {
      var predHTML = '<table style="border-collapse:collapse;">';
      var refHTML = '<table style="border-collapse:collapse;">';
      
      for (var i = 0; i < H; i++) {
        predHTML += '<tr>';
        refHTML += '<tr>';
        for (var j = 0; j < W; j++) {
          predHTML += '<td data-type="pred" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (predMask[i][j] ? '#4a90e2' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (predMask[i][j] ? 'white' : '#999') + ';">' + (predMask[i][j] ? '1' : '0') + '</td>';
          refHTML += '<td data-type="ref" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (refMask[i][j] ? '#f5a623' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (refMask[i][j] ? 'white' : '#999') + ';">' + (refMask[i][j] ? '1' : '0') + '</td>';
        }
        predHTML += '</tr>';
        refHTML += '</tr>';
      }
      
      predHTML += '</table>';
      refHTML += '</table>';
      
      predGridEl.innerHTML = predHTML;
      refGridEl.innerHTML = refHTML;
      
      // Adicionar event listeners
      predGridEl.querySelectorAll('td[data-type="pred"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          predMask[i][j] = predMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
      
      refGridEl.querySelectorAll('td[data-type="ref"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          refMask[i][j] = refMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
    }
    
    // Calcular métricas
    function calculateMetrics() {
      var TP = 0, FP = 0, FN = 0, TN = 0;
      
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          if (p && r) TP++;
          else if (p && !r) FP++;
          else if (!p && r) FN++;
          else TN++;
        }
      }
      
      var intersection = TP;
      var union = TP + FP + FN;
      var predCount = TP + FP;
      var refCount = TP + FN;
      
      var iou, dice;
      
      if (union === 0) {
        iou = 1.0;
        dice = 1.0;
      } else {
        iou = intersection / union;
        dice = (predCount + refCount === 0) ? 1.0 : (2 * intersection) / (predCount + refCount);
      }
      
      return { TP, FP, FN, TN, intersection, union, predCount, refCount, iou, dice };
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      var m = calculateMetrics();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var cellSize = Math.min(35, (canvas.width - 40) / W);
      var offsetX = (canvas.width - W * cellSize) / 2;
      var offsetY = (canvas.height - H * cellSize) / 2;
      
      // Desenhar grid
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          var color = '#f0f0f0';
          if (p && r) color = '#4a90e2';
          else if (p && !r) color = '#ff6b6b';
          else if (!p && r) color = '#f5a623';
          
          var x = offsetX + j * cellSize;
          var y = offsetY + i * cellSize;
          
          ctx.fillStyle = color;
          ctx.fillRect(x, y, cellSize - 2, cellSize - 2);
          ctx.strokeStyle = '#999';
          ctx.lineWidth = 1;
          ctx.strokeRect(x, y, cellSize - 2, cellSize - 2);
        }
      }
      
      // Fórmulas e cálculos
      var html = '';
      html += '<div style="margin-bottom:6px;"><b>1. Contagem de pixels:</b></div>';
      html += '<div style="color:#4a90e2;">TP (interseção) = ' + m.TP + '</div>';
      html += '<div style="color:#ff6b6b;">FP (só predita) = ' + m.FP + '</div>';
      html += '<div style="color:#f5a623;">FN (só referência) = ' + m.FN + '</div>';
      html += '<div style="color:#999;">TN (fundo) = ' + m.TN + '</div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>2. Interseção e União:</b></div>';
      html += '<div>Interseção = TP = <b>' + m.intersection + '</b></div>';
      html += '<div>União = TP + FP + FN = ' + m.TP + ' + ' + m.FP + ' + ' + m.FN + ' = <b>' + m.union + '</b></div>';
      html += '<div>|P| = TP + FP = ' + m.TP + ' + ' + m.FP + ' = <b>' + m.predCount + '</b></div>';
      html += '<div>|R| = TP + FN = ' + m.TP + ' + ' + m.FN + ' = <b>' + m.refCount + '</b></div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>3. Fórmulas:</b></div>';
      
      if (m.union === 0) {
        html += '<div style="color:#888;">IoU = 1.0 (máscaras vazias)</div>';
        html += '<div style="color:#888;">Dice = 1.0 (máscaras vazias)</div>';
      } else {
        html += '<div>IoU = Interseção / União = ' + m.intersection + ' / ' + m.union + ' = <b style="color:#2980b9;">' + m.iou.toFixed(4) + '</b></div>';
        html += '<div>Dice = 2·Interseção / (|P| + |R|) = 2·' + m.intersection + ' / (' + m.predCount + ' + ' + m.refCount + ') = ' + (2 * m.intersection) + ' / ' + (m.predCount + m.refCount) + ' = <b style="color:#50e3c2;">' + m.dice.toFixed(4) + '</b></div>';
      }
      
      metricsEl.innerHTML = html;
      
      // Resumo final
      summaryEl.innerHTML = '<b>IoU: ' + m.iou.toFixed(4) + '</b> &nbsp;&nbsp;|&nbsp;&nbsp; <b>Dice: ' + m.dice.toFixed(4) + '</b>';
    }
    
    // Limpar máscaras
    function clearMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = 0;
          refMask[i][j] = 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Gerar máscaras aleatórias
    function randomMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = Math.random() > 0.5 ? 1 : 0;
          refMask[i][j] = Math.random() > 0.5 ? 1 : 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Event listeners
    clearBtn.addEventListener('click', clearMasks);
    randomBtn.addEventListener('click', randomMasks);
    
    dimEl.addEventListener('input', function() {
      H = parseInt(dimEl.value);
      W = H;
      dimvEl.textContent = H + '×' + W;
      
      var newPred = [];
      var newRef = [];
      for (var i = 0; i < H; i++) {
        newPred.push([]);
        newRef.push([]);
        for (var j = 0; j < W; j++) {
          newPred[i].push(i < predMask.length && j < predMask[0].length ? predMask[i][j] : 0);
          newRef[i].push(i < refMask.length && j < refMask[0].length ? refMask[i][j] : 0);
        }
      }
      predMask = newPred;
      refMask = newRef;
      generateGrids();
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('quadrado');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0905');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.5:** Simulatore EP09_05: Valutazione della Segmentazione — IoU e Dice Pixel per Pixel


<figure id="fig-09-sim-ep0905">
  <img src="imagens/fig-09-sim-ep0905.png" alt=" Simulatore EP09_05: Valutazione della Segmentazione — IoU e Dice Pixel per Pixel " style="max-width:80%" />
  <figcaption><strong>Figura 9.5:</strong>  Simulatore EP09_05: Valutazione della Segmentazione — IoU e Dice Pixel per Pixel </figcaption>
</figure>

In [ ]:
%%writefile EP09_05.py
# Codice Python


In [ ]:
TestSuite("EP09_05.py").run()

### EP09_06 🔴 *Pipeline* Integrato: Dalla Rilevazione alla Misurazione nel Mondo Reale

Questo esercizio finale integra i due esercizi di rilevazione e il principio di **fotogrammetria** presentato nella sezione "Fotogrammetria e Riferimento di Scala" — esattamente lo stesso calcolo implementato nella figura di misurazione per riferimento di scala di questo capitolo. Lo scenario riproduce una situazione realistica: un rilevatore (Faster R-CNN o YOLO) genera **diverse scatole candidate sovrapposte** per lo stesso oggetto di interesse; dopo averle filtrate tramite NMS, la scatola sopravvissuta con maggiore confidenza viene utilizzata, insieme a una scatola di riferimento di larghezza reale nota (come la carta da $8{,}56$ cm), per stimare le dimensioni reali dell'oggetto rilevato.

#### 📋 Linee Guida di Implementazione

1. **Riferimento noto:** Leggere il valore reale $L_{ref}$ (larghezza reale dell'oggetto di riferimento, in cm) e successivamente i quattro reali $x_1\ y_1\ x_2\ y_2$ della sua scatola delimitante in pixel (già nota, senza necessità di rilevazione).
2. **Candidati dell'oggetto da misurare:** Leggere l'intero $N$ (numero di scatole candidate prodotte dal rilevatore per l'oggetto di interesse) e la soglia reale $\tau$; successivamente, leggere le $N$ righe di scatole candidate, ciascuna con $x_1\ y_1\ x_2\ y_2\ \text{score}$.
3. **Fase 1 — NMS:** Applicare esattamente l'algoritmo di Soppressione Non Massima dell'EP09_04 alle $N$ scatole candidate, utilizzando la soglia $\tau$, per eliminare rilevazioni ridondanti dello stesso oggetto.
4. **Fase 2 — Selezione della scatola finale:** Dopo il NMS, la scatola con il `score` più alto tra quelle mantenute è la rilevazione finale dell'oggetto (l'input garantisce che tutte le scatole candidate corrispondano a un singolo oggetto fisico, quindi la prima scatola selezionata dal NMS è già il risultato finale).
5. **Fase 3 — Misurazione per riferimento di scala:** Calcolare il rapporto $\text{cm/pixel} = L_{ref} / \text{larghezza del riferimento in pixel}$ e applicarlo sia alla larghezza che all'altezza (in pixel) della scatola finale dell'oggetto, ottenendo le sue dimensioni reali stimate in centimetri.
6. **Output:** Prima, una riga per ogni scatola mantenuta dopo il NMS (stesso formato dell'EP09_04): `índice score`. Successivamente, la riga `Total mantidas: X`. Infine, la riga `Objeto: L x A cm`, dove $L$ e $A$ sono la larghezza e l'altezza stimate dell'oggetto, ciascuna con 2 cifre decimali.

#### 📌 Vincoli Computazionali

* **Riutilizzare integralmente il NMS dell'EP09_04** — stessa regola di parità, stesso criterio di soppressione ($\text{IoU} > \tau$).
* **Il riferimento non passa attraverso il NMS:** la sua scatola è data direttamente, senza candidati concorrenti.
* **Rapporto unico per larghezza e altezza:** così come nella figura di fotogrammetria del capitolo, lo stesso rapporto cm/pixel (derivato dalla larghezza del riferimento) viene applicato sia alla larghezza che all'altezza dell'oggetto — non vi è calibrazione verticale separata.

#### 🧠 Fondamenti Teorici

| Fase | Concetto del capitolo |
|---|---|
| Multiple scatole candidate | Output grezzo di un rilevatore come Faster R-CNN o YOLO, prima della post-elaborazione |
| NMS (EP09_04) | Filtra le rilevazioni ridondanti, preservando solo la più affidabile per l'oggetto |
| Riferimento di scala noto | Stesso principio della carta da $8{,}56$ cm utilizzata nella sezione "Fotogrammetria e Riferimento di Scala" |
| Conversione pixel → centimetro | Regola del tre semplice: $\text{cm/pixel} = L_{ref} / w_{ref\_px}$, applicata alla scatola finale dell'oggetto |

#### 🧩 Metodi di `morph.py` che possono aiutare

* `mm.IoU(boxA, boxB)` — la stessa funzione suggerita nell'EP09_04, qui riutilizzata all'interno della fase di NMS di questo *pipeline* integrato (ricordarsi della conversione di formato: $w = x_2-x_1$, $h = y_2-y_1$).
* Se hai già risolto l'EP09_04 incapsulando il NMS in una funzione propria, questo è il momento ideale per **riutilizzare quel codice** — l'integrazione di moduli già testati singolarmente è esattamente la pratica ingegneristica che questo esercizio vuole rafforzare.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Reale $L_{ref}$.
* Riga 2: $x_1\ y_1\ x_2\ y_2$ della scatola di riferimento.
* Riga 3: Intero $N$ e reale $\tau$.
* Prossime $N$ righe: $x_1\ y_1\ x_2\ y_2\ \text{score}$ delle scatole candidate dell'oggetto.

**Output:**

* Una riga per ogni scatola mantenuta dopo il NMS: `índice score`.
* Riga successiva: `Total mantidas: X`.
* Ultima riga: `Objeto: L x A cm`.


#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 8.56<br>30 200 170 288<br>3 0.5<br>250 100 470 250 0.92<br>255 105 468 245 0.88<br>600 600 650 650 0.40 | 0 0.9200<br>2 0.4000<br>Total mantidas: 2<br>Objeto: 13.45 x 9.17 cm | La scatola 1 viene soppressa perché si sovrappone fortemente alla scatola 0; la rilevazione finale dell'oggetto è la scatola 0. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0906" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Pipeline Integrato — Dal Rilevamento alla Misurazione</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 Fotogrammetria</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Casella selezionata</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Casella soppressa</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Riferimento</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Oggetto finale</b>
      </span>
    </div>
    
    <!-- Controles -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">L_ref (cm)</label>
            <span id="ep0906_lref_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">8.56</span>
          </div>
          <input id="ep0906_lref" style="width:100%;accent-color:#2980b9;height:4px;" max="20" min="1" step="0.01" type="range" value="8.56">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Soglia τ</label>
            <span id="ep0906_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0906_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Esempio</label>
            <span id="ep0906_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Modello</span>
          </div>
          <select id="ep0906_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="padrao">Esempio Modello</option>
            <option value="multiplos">Oggetti Multipli</option>
            <option value="agrupado">Caselle Raggruppate</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0906_run_btn" style="background:#2980b9;color:white;border:none;padding:8px 16px;border-radius:16px;cursor:pointer;font-size:11px;font-weight:bold;transition:all 0.3s;">
            ▶️ Esegui Pipeline
          </button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0906_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;font-size:9px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:3fr 2fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:350px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Visualizzazione del Pipeline
        </div>
        <canvas id="ep0906_canvas" style="width:100%;height:300px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:350px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📋 Pipeline Passo dopo Passo
        </div>
        <div id="ep0906_steps" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resultado final -->
    <div id="ep0906_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var lrefEl = root.querySelector('#ep0906_lref');
    var lrefvEl = root.querySelector('#ep0906_lref_v');
    var tauEl = root.querySelector('#ep0906_tau');
    var tauvEl = root.querySelector('#ep0906_tau_v');
    var exampleEl = root.querySelector('#ep0906_example');
    var boxesConfigEl = root.querySelector('#ep0906_boxes_config');
    var canvas = root.querySelector('#ep0906_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0906_steps');
    var summaryEl = root.querySelector('#ep0906_summary');
    var runBtn = root.querySelector('#ep0906_run_btn');
    
    // Estado
    var refBox = { x1: 30, y1: 200, x2: 170, y2: 288 };
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    var finalBox = null;
    var cmPerPixel = 0;
    
    // Exemplos
    var examples = {
      padrao: {
        refBox: { x1: 30, y1: 200, x2: 170, y2: 288 },
        boxes: [
          { x1: 250, y1: 100, x2: 470, y2: 250, score: 0.92 },
          { x1: 255, y1: 105, x2: 468, y2: 245, score: 0.88 },
          { x1: 600, y1: 600, x2: 650, y2: 650, score: 0.40 }
        ]
      },
      multiplos: {
        refBox: { x1: 20, y1: 50, x2: 100, y2: 130 },
        boxes: [
          { x1: 200, y1: 150, x2: 350, y2: 280, score: 0.85 },
          { x1: 210, y1: 160, x2: 360, y2: 290, score: 0.75 },
          { x1: 400, y1: 300, x2: 550, y2: 420, score: 0.70 },
          { x1: 410, y1: 310, x2: 560, y2: 430, score: 0.65 }
        ]
      },
      agrupado: {
        refBox: { x1: 50, y1: 50, x2: 150, y2: 150 },
        boxes: [
          { x1: 300, y1: 200, x2: 500, y2: 350, score: 0.95 },
          { x1: 310, y1: 210, x2: 490, y2: 340, score: 0.90 },
          { x1: 320, y1: 220, x2: 480, y2: 330, score: 0.85 },
          { x1: 330, y1: 230, x2: 470, y2: 320, score: 0.80 }
        ]
      }
    };
    
    // Ajustar canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      refBox = JSON.parse(JSON.stringify(example.refBox));
      boxes = JSON.parse(JSON.stringify(example.boxes));
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Processar Pipeline" para executar.';
      summaryEl.innerHTML = 'Aguardando processamento...';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      html += '<div style="background:#e8f5e9;border:1px solid #a5d6a7;border-radius:6px;padding:4px 6px;">';
      html += '<b>Referência:</b> ';
      html += '<input type="number" id="ep0906_ref_x1" value="' + refBox.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y1" value="' + refBox.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_x2" value="' + refBox.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y2" value="' + refBox.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '</div>';
      
      boxes.forEach(function(box, i) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;">';
        html += '<b>#' + i + ':</b> ';
        html += '<input type="number" id="ep0906_x1_' + i + '" value="' + box.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y1_' + i + '" value="' + box.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_x2_' + i + '" value="' + box.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y2_' + i + '" value="' + box.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_score_' + i + '" value="' + box.score + '" step="0.05" min="0" max="1" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '</div>';
      });
      
      boxesConfigEl.innerHTML = html;
      
      // Event listeners para referência
      ['x1', 'y1', 'x2', 'y2'].forEach(function(field) {
        var input = root.querySelector('#ep0906_ref_' + field);
        if (input) {
          input.addEventListener('input', function() {
            refBox[field] = parseFloat(input.value) || 0;
            render();
          });
        }
      });
      
      // Event listeners para caixas
      boxes.forEach(function(box, i) {
        ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
          var input = root.querySelector('#ep0906_' + field + '_' + i);
          if (input) {
            input.addEventListener('input', function() {
              boxes[i][field] = parseFloat(input.value) || 0;
              selectedBoxes = [];
              suppressedBoxes = [];
              finalBox = null;
              render();
            });
          }
        });
      });
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar pipeline
    function runPipeline() {
      var tau = parseFloat(tauEl.value);
      var lref = parseFloat(lrefEl.value);
      
      // Etapa 1: NMS
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) return b.box.score - a.box.score;
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      steps.push('<div style="font-weight:bold;color:#333;">Etapa 1: NMS</div>');
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push('<div style="color:#4a90e2;">Selecionar caixa #' + selected.originalIndex + ' (score: ' + selected.box.score.toFixed(4) + ')</div>');
        
        var newRemaining = [];
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressedBoxes.push(remaining[i]);
            steps.push('<div style="color:#ff6b6b;margin-left:10px;">↳ Suprimir #' + remaining[i].originalIndex + ' (IoU: ' + iou.toFixed(3) + ')</div>');
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        remaining = newRemaining;
      }
      
      steps.push('<div style="margin-top:4px;">Total mantidas: <b>' + selectedBoxes.length + '</b></div>');
      
      // Etapa 2: Seleção da caixa final
      if (selectedBoxes.length > 0) {
        finalBox = selectedBoxes[0];
        steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 2: Caixa Final</div>');
        steps.push('<div>Caixa selecionada: #' + finalBox.originalIndex + ' (score: ' + finalBox.box.score.toFixed(4) + ')</div>');
      }
      
      // Etapa 3: Medição
      steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 3: Medição por Referência</div>');
      
      var refWidthPx = refBox.x2 - refBox.x1;
      cmPerPixel = lref / refWidthPx;
      
      steps.push('<div>Largura da referência: ' + refWidthPx + ' pixels</div>');
      steps.push('<div>cm/pixel = ' + lref + ' / ' + refWidthPx + ' = <b>' + cmPerPixel.toFixed(6) + '</b></div>');
      
      if (finalBox) {
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        steps.push('<div>Largura do objeto: ' + objWidthPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objWidthCm.toFixed(2) + ' cm</b></div>');
        steps.push('<div>Altura do objeto: ' + objHeightPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objHeightCm.toFixed(2) + ' cm</b></div>');
        
        stepsEl.innerHTML = steps.join('');
        
        summaryEl.innerHTML = '<b>Objeto: ' + objWidthCm.toFixed(2) + ' x ' + objHeightCm.toFixed(2) + ' cm</b>';
      } else {
        stepsEl.innerHTML = steps.join('');
        summaryEl.innerHTML = 'Nenhum objeto detectado.';
      }
      
      render();
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var scale = Math.min(canvas.width / 700, canvas.height / 700);
      var offsetX = 20;
      var offsetY = 20;
      
      // Desenhar caixa de referência
      drawBoxOnCanvas(refBox, '#50e3c2', 'Ref', scale, offsetX, offsetY);
      
      // Desenhar caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        var isFinal = finalBox && finalBox.originalIndex === index;
        
        var color = '#f5a623';
        var label = '#' + index;
        
        if (isFinal) {
          color = '#f5a623';
          label = '#' + index + ' ✓';
        } else if (isSelected) {
          color = '#4a90e2';
        } else if (isSuppressed) {
          color = '#ff6b6b';
          label = '#' + index + ' ✗';
        }
        
        drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY);
      });
      
      // Desenhar linhas de medição
      if (finalBox && cmPerPixel > 0) {
        var refWidthPx = refBox.x2 - refBox.x1;
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        // Linha de largura do objeto
        var y = offsetY + finalBox.box.y1 * scale - 10;
        var x1 = offsetX + finalBox.box.x1 * scale;
        var x2 = offsetX + finalBox.box.x2 * scale;
        
        ctx.strokeStyle = '#f5a623';
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(x1, y);
        ctx.lineTo(x2, y);
        ctx.stroke();
        
        ctx.fillStyle = '#f5a623';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(objWidthCm.toFixed(2) + ' cm', (x1 + x2) / 2, y - 3);
      }
    }
    
    // Desenhar caixa no canvas
    function drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY) {
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      ctx.fillStyle = color;
      ctx.font = 'bold 11px Arial';
      ctx.textAlign = 'center';
      ctx.fillText(label, x + w/2, y - 5);
    }
    
    // Event listeners
    runBtn.addEventListener('click', function() {
      runPipeline();
      runBtn.textContent = '✓ Processado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Processar Pipeline';
        runBtn.style.background = '#2980b9';
      }, 1000);
    });
    
    lrefEl.addEventListener('input', function() {
      lrefvEl.textContent = parseFloat(lrefEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0906');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.6:** Simulatore EP09_06: Pipeline Integrato — Rilevamento alla Misurazione del Mondo Reale


<figure id="fig-09-sim-ep0906">
  <img src="imagens/fig-09-sim-ep0906.png" alt=" Simulatore EP09_06: Pipeline Integrato — Rilevamento alla Misurazione del Mondo Reale " style="max-width:80%" />
  <figcaption><strong>Figura 9.6:</strong>  Simulatore EP09_06: Pipeline Integrato — Rilevamento alla Misurazione del Mondo Reale </figcaption>
</figure>

In [ ]:
%%writefile EP09_06.py
# Codice Python

In [ ]:
TestSuite("EP09_06.py").run()